# Subway disruptions and FHV pickups in NYC

When subway service is lost, do more people take an FHV from the affected area? My forecasting work describes how pickups vary across zones and hours. Here I want to understand whether sustained subway service loss is followed by additional completed pickups, compared with what I would expect without the disruption.

I compare affected zones with similar zones at the same actual hours. The panel covers July 2024–June 2026, and effect estimation starts in September 2024. Each event uses its own earlier eight-week history. This analysis is part of my NYC Mobility Digital Twin capstone.

My main estimate uses matched difference-in-differences after historical weekday–hour and linear weather adjustment. I keep the other specifications as diagnostics and supporting comparisons. These choices developed during exploratory work; this is not a preregistered analysis.

In this saved run, the primary contrast is **18.06 additional completed pickups per hour**, with a reported incident-clustered 95% interval of **12.06 to 24.05**, across 368 episode–zone pairs and 163 incidents. The results are consistent with increased pickups during the selected sustained-loss periods, but do not identify passengers who switched from the subway.

I include clock hours with at least 45 minutes of measured loss only when their qualifying spell also contains a full clock hour. Standalone 45-minute spells remain excluded. Dropoffs, duration effects and after-recovery effects are outside the completed analysis. Interpretations below describe the saved outputs and must be reviewed if the notebook is rerun with different inputs or rules.

## 1. Defining the comparison

A disruption-hour pickup count alone cannot tell me the effect. Rain, commuting patterns or a nearby event could increase pickups even without service loss. I need a comparison for the change that would otherwise have happened.

For example, suppose pickups rise from 100 to 150 in the affected zone and from 80 to 100 in its comparison. The contrast is $(150-100)-(100-80)=30$ pickups per hour. The example explains the calculation; these are not observed results.

Baseline and treatment hours often occur at different clock times. Even well-matched zones may have different daily curves, so I also subtract their usual historical weekday–hour gap. Each episode–zone pair receives equal weight; an incident affecting several zones contributes several pairs.

| Decision | Rule used here |
|---|---|
| Outcome | Completed high-volume FHV pickups, `total_pickups` |
| Episodes | Selected causes, nonholiday weekday daytime alert onset, observed closing update, successful usable measurement and no recorded nearby alert gap |
| Sustained loss | At least four consecutive 15-minute loss bins in one episode–route–direction–complex series; partial and complete route loss can join |
| Treatment hours | At least 45 minutes of overlap with one qualifying spell that contains a complete clock hour; quality and attribution checks still apply; nonholiday weekdays 06:00–19:59 |
| Baseline | Three complete hours before the earlier of first attributed zone exposure and first recorded incident alert |
| Comparisons | Same borough, excluding self, shared station complexes, same-episode exposure and recorded exposure or quality problems throughout the comparison window |
| Matching | Earlier eight weeks; at least 280 common usable hours and 50% coverage; comparison/treated mean ratio 0.5–2.0; up to three lowest count-RMSE comparisons |
| Censored episodes | Not treatments; mapped hours blocked from first alert through 48 hours after the last active update |
| Primary estimate | Historical weekday–hour and linear weather adjustment; one common contrast; incident-clustered uncertainty |
| Supporting checks | Borough estimates, event study, placebo dates, adjacent-zone exclusion, influence checks and same-zone models |

These thresholds are working research choices. Unknown exposure stays unknown. Route loss does not necessarily mean the whole station complex has no subway access, and the 48-hour buffer does not establish recovery.

## 2. Setting up the analysis

I keep paths, dates, matching rules and feature definitions in one configuration cell. This makes it easier to see what the analysis actually uses. If I change a rule, I need to restart and run from the top so later tables do not mix old and new objects.

The helpers below handle local time, weather scaling, table display and clustered summaries. They support the calculations; the research decisions remain visible in the configuration and explanations.

I run this from the showcase with the prepared data downloaded and the Conda environment active. The [setup instructions](../README.md#run-the-analysis-locally) show how to get both. If I'm running without a CUDA GPU, I set `RUN_XGBOOST_APPENDIX = False` to skip the optional same-zone model comparison.


In [ ]:
# Loading tools used by this analysis
import json
from datetime import timedelta
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import trim_mean
from IPython.display import HTML, Markdown, display

# Reporting dependencies without installing or upgrading the environment
package_names = ["pandas", "numpy", "pyarrow", "duckdb", "statsmodels", "scipy",
                 "matplotlib", "seaborn", "geopandas", "xgboost", "scikit-learn"]


def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


display(pd.DataFrame({"Package": package_names,
                      "Version": [installed_version(p) for p in package_names]}))

In [ ]:
# Defining project paths and the implemented research rules
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src" / "ride_demand").is_dir())
MTA_RUN_DIR = PROJECT_ROOT / "data/silver/mta_disruptions/full_period_3_5_20261012_034129"
TAXI_ZONE_SHAPEFILE = PROJECT_ROOT / "data/external/tlc_zones.geojson"

LOCAL_TIMEZONE = "America/New_York"
PANEL_START = pd.Timestamp("2024-07-01")
ANALYSIS_START = pd.Timestamp("2024-09-01")
STUDY_END_EXCLUSIVE = pd.Timestamp("2026-07-01")
EXCLUDED_ZONE_IDS = {1, 104, 105}

# Matching rules
MATCH_HISTORY_WEEKS = 8
MAX_COMPARISONS = 3
MIN_SHARED_HISTORY_HOURS = 280
MIN_SHARED_HISTORY_SHARE = 0.50
MIN_VOLUME_RATIO, MAX_VOLUME_RATIO = 0.50, 2.00
CENSORED_BUFFER_HOURS = 48

# Diagnostics
EVENT_STUDY_HOURS = list(range(-6, 4))      # relative to the baseline anchor hour
PLACEBO_MAX_SHIFT_WEEKS = 12                # placebo dates up to 12 weeks before/after
ADJACENCY_BUFFER_FT = 50                    # polygons within 50 ft count as adjacent
BOOTSTRAP_DRAWS = 2000
RANDOM_SEED = 42

# Shared feature definitions (one copy for every model in this notebook)
WEATHER_SCALES = {"rain_in": 0.1, "snowfall_in": 1.0, "wind_speed_mph": 5.0, "temperature_f": 10.0}
EVENT_FLAGS = ["is_unofficial_holiday", "is_post_game_demand", "is_concert_window", "is_wide_concert_window"]
FEATURE_SCALES = {**WEATHER_SCALES, **{flag: 1.0 for flag in EVENT_FLAGS}}
REFERENCE_FEATURES = list(FEATURE_SCALES)

# Appendix: XGBoost sensitivity analysis
RUN_XGBOOST_APPENDIX = True
XGB_BASE_PARAMS = {"objective": "reg:squarederror", "tree_method": "hist", "device": "cuda",
                   "n_estimators": 200, "learning_rate": 0.05, "reg_lambda": 10,
                   "n_jobs": 4, "random_state": RANDOM_SEED}
XGB_GRID = {"max_depth": [3, 5], "min_child_weight": [5, 20]}
XGB_VALIDATION_WEEKS = 2

PAIR_KEYS = ["episode_key", "LocationID"]
pair_keys = PAIR_KEYS  # name used inside the original stage functions

input_paths = {
    "mta_manifest": MTA_RUN_DIR / "run_manifest.json",
    "mta_hourly_panel": MTA_RUN_DIR / "hourly_panel.parquet",
    "mta_episodes": MTA_RUN_DIR / "disruption_episodes.parquet",
    "mta_attributed_bins": MTA_RUN_DIR / "attributed_bins.parquet",
    "mta_station_zone_bridge": MTA_RUN_DIR / "station_complex_zone_bridge.parquet",
    "uncertain_windows": MTA_RUN_DIR / "uncertain_announced_windows.parquet",
    "fhv_weather_calendar_events": PROJECT_ROOT / "data/gold/zone_hour_pickups_weather_sports_concerts.parquet",
}
assert PANEL_START < ANALYSIS_START < STUDY_END_EXCLUSIVE
assert ANALYSIS_START.date() - timedelta(days=7 * MATCH_HISTORY_WEEKS) >= PANEL_START.date()
assert PLACEBO_MAX_SHIFT_WEEKS > MATCH_HISTORY_WEEKS
missing_inputs = [str(p) for p in input_paths.values() if not p.is_file()]
assert not missing_inputs, "Missing project inputs: " + ", ".join(missing_inputs)

In [ ]:
# Setting consistent fonts, colors, and spacing
sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette=["#10707B", "#4477AA", "#CC6677", "#CCAA66"],
    rc={
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 11,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.titlelocation": "left",
        "axes.titleweight": "normal",
        "axes.titlesize": 15,
        "axes.titlepad": 16,
        "axes.labelsize": 11,
        "axes.labelpad": 10,
        "axes.edgecolor": "#B0B0B0",
        "axes.linewidth": 0.7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.axisbelow": True,
        "text.color": "#252525",
        "axes.labelcolor": "#404040",
        "xtick.color": "#555555",
        "ytick.color": "#555555",
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "grid.color": "#E5E7EB",
        "grid.linewidth": 0.6,
        "legend.frameon": False,
        "legend.fontsize": 10,
    },
)


# Displaying readable tables without changing the underlying data
def display_table(frame, decimals=2, caption=None, labels=None):
    """Displaying report-friendly headings, numbers, and total rows."""

    # Defining readable labels for commonly used research columns
    default_labels = {
        "zone_borough": "Borough",
        "specification": "Specification",
        "episode_zone_pairs": "Episode–zone pairs",
        "incidents": "Incidents",
        "share_of_pairs_pct": "Share of pairs (%)",
        "contrast_pickups_per_hour": "Pickup contrast per hour",
        "clustered_se": "Clustered SE",
        "estimate": "Estimate",
        "ci95_lower": "95% CI lower",
        "ci95_upper": "95% CI upper",
        "matched_pairs": "Matched pairs",
        "fully_covered_pct": "Fully covered (%)",
    }
    heading_labels = {**default_labels, **(labels or {})}

    # Converting other variable names into readable fallback headings
    def readable(name):
        if name is None:
            return ""
        return heading_labels.get(
            name, str(name).replace("_", " ").capitalize()
        )

    # Creating a display copy while preserving the original table
    table = frame.copy()
    table.columns = table.columns.map(readable)
    table.index = table.index.set_names(
        [readable(name) for name in table.index.names]
    )
    assert table.columns.is_unique, "Display headings must be unique."

    formats = {
        column: (
            "{:,.0f}"
            if pd.api.types.is_integer_dtype(table[column])
            else f"{{:,.{decimals}f}}"
        )
        for column in table.select_dtypes(include="number").columns
    }

    # Highlighting totals in ordinary and multi-level indexes
    def emphasize_total(row):
        labels = row.name if isinstance(row.name, tuple) else (row.name,)
        style = (
            "font-weight: 700; background-color: #EDF4F5;"
            if "Total" in labels else ""
        )
        return [style] * len(row)

    styled = (
        table.style
        .format(formats, na_rep="—")
        .set_properties(**{
            "background-color": "#FFFFFF",
            "color": "#252525",
            "text-align": "center",
            "padding": "6px 8px",
            "border-bottom": "1px solid #E5E7EB",
        })
        .set_table_styles([
            {
                "selector": "",
                "props": [
                    ("font-family", "Arial, DejaVu Sans, sans-serif"),
                    ("font-size", "12px"),
                    ("background-color", "#FFFFFF"),
                    ("color", "#252525"),
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#F3F5F7"),
                    ("color", "#252525"),
                    ("font-weight", "600"),
                    ("text-align", "center"),
                    ("white-space", "normal"),
                    ("overflow-wrap", "anywhere"),
                    ("padding", "7px 8px"),
                    ("border-bottom", "1px solid #CBD2D9"),
                ],
            },
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("background-color", "#FFFFFF"),
                    ("color", "#252525"),
                    ("text-align", "left"),
                    ("font-size", "13px"),
                    ("font-weight", "600"),
                    ("padding", "8px"),
                ],
            },
        ])
        .apply(emphasize_total, axis=1)
    )

    if caption:
        styled = styled.set_caption(caption)

    display(styled)

In [ ]:
# Shared helpers for time zones, clustering, weather terms and clustered means
def to_local(series):
    """Converting UTC timestamps to naive New York clock labels used by the panel."""
    return series.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)


def cluster_groups(data, how="incident"):
    """Returning cluster codes: by incident, or two-way by incident and treated zone."""
    incident = pd.factorize(data["event_id"])[0]
    if how == "incident":
        return incident
    if how == "two_way":
        return np.column_stack([incident, pd.factorize(data["LocationID"])[0]])
    raise ValueError(f"Unknown clustering: {how}")


def clustered_mean_summary(data, outcome, estimate_name="estimate",
                           cluster="incident", by_borough=True):
    """Estimating equal-row means by borough and overall, with clustered intervals."""
    assert data[outcome].notna().all()
    samples = dict(tuple(data.groupby("zone_borough"))) if by_borough else {}
    samples["Total"] = data
    rows = []
    for borough, sample in samples.items():
        row = {"zone_borough": borough,
               "episode_zone_pairs": len(sample),
               "incidents": sample["event_id"].nunique(),
               "share_of_pairs_pct": 100 * len(sample) / len(data),
               estimate_name: sample[outcome].mean(),
               "clustered_se": np.nan, "ci95_lower": np.nan, "ci95_upper": np.nan}
        # A clustered interval needs at least two incidents
        if sample["event_id"].nunique() > 1:
            model = smf.ols(f"{outcome} ~ 1", data=sample, missing="raise").fit(
                cov_type="cluster",
                cov_kwds={"groups": cluster_groups(sample, cluster),
                          "use_correction": True, "df_correction": True},
                use_t=True)
            assert np.isclose(model.params["Intercept"], row[estimate_name])
            lower, upper = model.conf_int().loc["Intercept"]
            row.update(clustered_se=model.bse["Intercept"], ci95_lower=lower, ci95_upper=upper)
        rows.append(row)
    return pd.DataFrame(rows).set_index("zone_borough")


def add_weather_terms(frame):
    """Adding scaled treated and comparison weather columns; returning their names."""
    terms = []
    for prefix in ["treated", "comparison"]:
        for source, scale in WEATHER_SCALES.items():
            column = f"{prefix}_{source}_scaled"
            frame[column] = frame[f"{prefix}_{source}"] / scale
            terms.append(column)
    return terms

## 3. Connecting the existing data

I already have hourly pickup counts with weather, calendar and event features. I will reuse that table and join the verified MTA panel by taxi zone and New York local hour. Before interpreting any result, I want to check that the join neither loses observations nor creates duplicate outcomes.

Zones 1, 104 and 105 are excluded. I keep the zone-level observations even when the report later summarizes results by borough.

The [subway measurement notebook](03_mta_disruptions_measured_v3_1.ipynb) explains how I turn train records into service-loss periods. In the [validation notebook](04_mta_disruptions_validation.ipynb), I check examples against the recorded train service. Here I use the saved measurement tables to ask what happened to pickups.


In [ ]:
def load_research_panel():
    """Loading and validating the outcome-exposure join without changing saved data."""
    with input_paths['mta_manifest'].open(encoding='utf-8') as file:
        mta_manifest = json.load(file)

    # Selecting the candidate FHV input
    FHV_PATH = input_paths["fhv_weather_calendar_events"]

    # Checking coverage and values without loading the full dataset into pandas
    with duckdb.connect() as audit_con:
        fhv_audit = audit_con.execute(
            """
            WITH research_rows AS (
                SELECT *
                FROM read_parquet(?)
                WHERE hour >= ?
                  AND hour < ?
                  AND location_id NOT IN (?, ?, ?)
            )
            SELECT
                COUNT(*) AS rows,
                COUNT(DISTINCT (location_id, hour)) AS unique_zone_hours,
                COUNT(*) - COUNT(DISTINCT (location_id, hour))
                    AS duplicate_key_rows,
                COUNT(DISTINCT location_id) AS zones,
                COUNT(DISTINCT hour) AS distinct_hours,
                MIN(hour) AS first_hour,
                MAX(hour) AS last_hour,
                COUNT(*) FILTER (WHERE total_pickups IS NULL)
                    AS missing_pickups,
                COUNT(*) FILTER (WHERE total_pickups < 0)
                    AS negative_pickups,
                COUNT(*) FILTER (
                    WHERE temperature_f IS NULL
                       OR precipitation_in IS NULL
                       OR rain_in IS NULL
                       OR snowfall_in IS NULL
                       OR wind_speed_mph IS NULL
                ) AS rows_missing_weather
            FROM research_rows
            """,
            [
                str(FHV_PATH),
                PANEL_START.to_pydatetime(),
                STUDY_END_EXCLUSIVE.to_pydatetime(),
                *sorted(EXCLUDED_ZONE_IDS),
            ],
        ).df()

    # Checking whether every eligible MTA zone-hour has an FHV match
    with duckdb.connect() as audit_con:
        join_audit = audit_con.execute(
            """
            WITH mta AS (
                SELECT LocationID, hour_local
                FROM read_parquet(?)
                WHERE hour_local >= ?
                  AND hour_local < ?
                  AND LocationID NOT IN (?, ?, ?)
            ),
            fhv AS (
                SELECT location_id, hour, TRUE AS fhv_row_present
                FROM read_parquet(?)
            )
            SELECT
                COUNT(*) AS joined_rows,
                COUNT(DISTINCT (m.LocationID, m.hour_local))
                    AS unique_mta_zone_hours,
                COUNT(DISTINCT m.LocationID) AS mta_zones,
                COUNT(*) FILTER (WHERE f.fhv_row_present IS TRUE)
                    AS matched_rows,
                COUNT(*) FILTER (WHERE f.fhv_row_present IS NULL)
                    AS unmatched_rows
            FROM mta AS m
            LEFT JOIN fhv AS f
                ON m.LocationID = f.location_id
               AND m.hour_local = f.hour
            """,
            [
                str(input_paths["mta_hourly_panel"]),
                PANEL_START.to_pydatetime(),
                STUDY_END_EXCLUSIVE.to_pydatetime(),
                *sorted(EXCLUDED_ZONE_IDS),
                str(FHV_PATH),
            ],
        ).df()

    # Combining the audited inputs while retaining all MTA exposure flags
    with duckdb.connect() as panel_con:
        research_panel = panel_con.execute(
            """
            SELECT
                m.*,
                f.* EXCLUDE (location_id, hour)
            FROM read_parquet(?) AS m
            LEFT JOIN read_parquet(?) AS f
                ON m.LocationID = f.location_id
               AND m.hour_local = f.hour
            WHERE m.hour_local >= ?
              AND m.hour_local < ?
              AND m.LocationID NOT IN (?, ?, ?)
            """,
            [
                str(input_paths["mta_hourly_panel"]),
                str(FHV_PATH),
                PANEL_START.to_pydatetime(),
                STUDY_END_EXCLUSIVE.to_pydatetime(),
                *sorted(EXCLUDED_ZONE_IDS),
            ],
        ).df()

    # Validating that the join preserves the audited observations
    expected_rows = int(join_audit.loc[0, "unique_mta_zone_hours"])

    assert len(research_panel) == expected_rows
    assert not research_panel.duplicated(["LocationID", "hour_local"]).any()
    assert research_panel["total_pickups"].notna().all()
    assert research_panel["exposure_known"].notna().all()

    # Summarizing the combined panel
    assert pd.Timestamp(mta_manifest['study_start']).tz_convert(LOCAL_TIMEZONE).tz_localize(None) == PANEL_START
    assert pd.Timestamp(mta_manifest['study_end_exclusive']).tz_convert(LOCAL_TIMEZONE).tz_localize(None) == STUDY_END_EXCLUSIVE
    assert research_panel['hour_local'].min() == PANEL_START
    assert research_panel['hour_local'].max() < STUDY_END_EXCLUSIVE

    return research_panel, mta_manifest, fhv_audit, join_audit

In [ ]:
research_panel, mta_manifest, fhv_audit, join_audit = load_research_panel(
)

display_table(join_audit.T.rename(columns={0: "Value"}))

The join retains all **2,890,800 zone-hours across 165 subway-served zones**, with no unmatched outcome keys. The row count agrees with the unique MTA zone-hour count, so this join did not multiply observations.

This gives me a consistent outcome panel. It does not independently verify every disruption measurement. Next, I need to distinguish observed clean hours from hours where exposure or pickups are uncertain.

## 4. Keeping uncertain and altered observations out

Unknown exposure is not evidence of normal service. I keep the underlying records and quality flags, but exclude uncertain observations from the relevant treatment, baseline and comparison samples.

The engineering pipeline replaced some repeated November 1 AM pickup counts using neighboring weeks. Those values are not directly observed research outcomes. I exclude those altered hours and nonexistent spring-forward hours.

For unresolved episodes, the saved three-hour endpoint is a processing cutoff. I block mapped zone-hours from the first recorded incident alert through 48 hours after the last active update. This is a working protection against contaminated controls, not an observed recovery time. The buffer and other quality exclusions can overlap.

In [ ]:
def apply_research_quality(research_panel):
    """Applying recorded quality, DST, and working censored-episode exclusions once."""
    # Checking the reasons used to mark exposure as unknown
    exposure_reason_cols = [
        "alert_gap",
        "train_data_missing",
        "feed_problem",
        "unmeasured_episode",
        "censored_alert_state",
    ]

    assert research_panel[exposure_reason_cols].notna().all().all()

    unknown_from_reasons = research_panel[exposure_reason_cols].any(axis=1)

    assert (
        research_panel["exposure_known"] == ~unknown_from_reasons
    ).all(), "Exposure status disagrees with its recorded reasons."

    # Summarizing overlapping quality flags and DST indicators
    quality_cols = exposure_reason_cols + [
        "ambiguous_dst_hour",
        "nonexistent_dst_hour",
    ]

    quality_summary = pd.DataFrame(
        {
            "flagged_zone_hours": research_panel[quality_cols].sum(),
            "share_of_panel": research_panel[quality_cols].mean(),
        }
    )


    # Comparing exposure coverage across boroughs
    borough_coverage = (
        research_panel.groupby("zone_borough", dropna=False)
        .agg(
            zone_hours=("exposure_known", "size"),
            known_zone_hours=("exposure_known", "sum"),
            known_share=("exposure_known", "mean"),
        )
    )

    borough_coverage["unknown_zone_hours"] = (
        borough_coverage["zone_hours"] - borough_coverage["known_zone_hours"]
    )

    # Loading the records needed to construct censored-episode exclusions
    episode_records = pd.read_parquet(input_paths["mta_episodes"])
    uncertain_windows = pd.read_parquet(input_paths["uncertain_windows"])

    # The working buffer length comes from the configuration cell
    # Retaining mapped zones for censored episodes, including context episodes
    censored_zone_windows = (
        uncertain_windows[["episode_key", "LocationID"]]
        .drop_duplicates()
        .merge(
            episode_records.loc[
                episode_records["is_right_censored"].eq(True),
                ["episode_key", "incident_start_uncapped", "last_active_time"],
            ],
            on="episode_key",
            how="inner",
            validate="many_to_one",
        )
    )

    assert censored_zone_windows[
        ["incident_start_uncapped", "last_active_time"]
    ].notna().all().all()

    # Converting exclusion boundaries to local-hour labels conservatively
    censored_zone_windows["start_local"] = (
        censored_zone_windows["incident_start_uncapped"]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
        .dt.floor("h")
    )

    censored_zone_windows["end_local"] = (
        (
            censored_zone_windows["last_active_time"]
            + pd.DateOffset(hours=CENSORED_BUFFER_HOURS)
        )
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
        .dt.ceil("h")
    )

    # Flagging any panel hour overlapping a mapped exclusion window
    panel_keys = research_panel[["LocationID", "hour_local"]]

    with duckdb.connect() as exclusion_con:
        exclusion_con.register("panel_keys", panel_keys)
        exclusion_con.register("censored_windows", censored_zone_windows)

        censor_flags = exclusion_con.execute(
            """
            SELECT
                p.LocationID,
                p.hour_local,
                EXISTS (
                    SELECT 1
                    FROM censored_windows AS c
                    WHERE c.LocationID = p.LocationID
                      AND p.hour_local >= c.start_local
                      AND p.hour_local < c.end_local
                ) AS censored_episode_buffer
            FROM panel_keys AS p
            """
        ).df()

    # Preserving row order and allowing this cell to be rerun
    research_panel = (
        research_panel.drop(columns="censored_episode_buffer", errors="ignore")
        .merge(
            censor_flags,
            on=["LocationID", "hour_local"],
            how="left",
            validate="one_to_one",
            sort=False,
        )
    )

    assert research_panel["censored_episode_buffer"].notna().all()

    # Reporting the flag's footprint without changing the existing estimates
    censor_buffer_summary = research_panel.groupby("zone_borough").agg(
        total_zone_hours=("censored_episode_buffer", "size"),
        buffered_zone_hours=("censored_episode_buffer", "sum"),
    )

    censor_buffer_summary.loc["Total"] = censor_buffer_summary.sum()
    censor_buffer_summary["buffered_pct"] = (
        100
        * censor_buffer_summary["buffered_zone_hours"]
        / censor_buffer_summary["total_zone_hours"]
    )

    # Identifying hours targeted by the engineering pipeline's replacement rule
    altered_pickup_hours = pd.to_datetime(
        ["2024-11-03 01:00:00", "2025-11-02 01:00:00"]
    )

    research_panel["pickup_altered_by_pipeline"] = (
        research_panel["hour_local"].isin(altered_pickup_hours)
    )

    # Checking agreement with the independently recorded MTA DST flag
    assert research_panel["pickup_altered_by_pipeline"].equals(
        research_panel["ambiguous_dst_hour"]
    )

    # Flagging hours unsuitable for interpreting pickups as a single observed hour
    research_panel["outcome_hour_excluded"] = (
        research_panel["pickup_altered_by_pipeline"]
        | research_panel["ambiguous_dst_hour"]
        | research_panel["nonexistent_dst_hour"]
    )

    # Requiring known exposure, valid outcome hours, and no censored-episode buffer
    research_panel["passes_initial_quality"] = (
        research_panel["exposure_known"]
        & ~research_panel["outcome_hour_excluded"]
        & ~research_panel["censored_episode_buffer"]
    )

    # Summarizing coverage after overlapping exclusions
    initial_quality_summary = (
        research_panel.groupby("zone_borough")
        .agg(
            total_zone_hours=("passes_initial_quality", "size"),
            altered_pickup_hours=("pickup_altered_by_pipeline", "sum"),
            excluded_outcome_hours=("outcome_hour_excluded", "sum"),
            passing_zone_hours=("passes_initial_quality", "sum"),
            passing_share=("passes_initial_quality", "mean"),
        )
    )

    return research_panel, episode_records, uncertain_windows, quality_summary, censor_buffer_summary, initial_quality_summary

In [ ]:
research_panel, episode_records, uncertain_windows, quality_summary, censor_buffer_summary, initial_quality_summary = apply_research_quality(
    research_panel,
)

display_table(censor_buffer_summary)

The censor buffer marks **522,876 zone-hours, or 18.09% of the panel**. Manhattan has the largest buffered share at 24.78%, compared with 17.28% in Brooklyn, 15.16% in Queens and 14.27% in the Bronx. These are buffer counts, not additional nonoverlapping losses on top of every other exclusion.

This matters because usable disruptions and comparison histories may differ across boroughs. I will keep the buffer rather than treat unresolved hours as clean, but the final findings will describe the retained sample, not every disruption in NYC.

## 5. Finding sustained loss and eligible pickup hours

An alert lasting an hour does not establish that service was lost for that hour. I reconstruct uninterrupted spells from measured 15-minute bins, holding episode, route, direction and station complex fixed. Partial and complete route loss can form one spell; diversion alone does not qualify.

I require at least 60 continuous minutes and at least one complete clock hour within the spell. From those spells, I retain clock hours containing at least 45 minutes of loss. I do not add short losses from different spells or routes to reach the threshold.

For loss from 10:15 to 12:15, the 10:00–11:00 hour has 45 minutes of loss and the 11:00–12:00 hour has 60; both can enter. The 12:00–13:00 hour has only 15 minutes and is excluded. Loss from 10:15 to 11:15 remains excluded because it contains no complete clock hour. Standalone 45-minute spells also remain excluded.

Pickups still cover the whole clock hour. I do not scale them by exposure minutes. Measurement-window boundaries and shared attribution are separate quality concerns; a 45-minute clock-hour overlap does not waive those checks.

In [ ]:
def screen_episodes():
    """Screening alert episodes while retaining all records for contamination checks."""
    # Loading episode records while retaining context and excluded episodes
    episode_records = pd.read_parquet(input_paths["mta_episodes"])

    assert episode_records["episode_key"].notna().all()
    assert episode_records["episode_key"].is_unique
    assert episode_records["onset_time"].notna().all()

    # Comparing UTC episode timestamps with timezone-aware study boundaries
    analysis_start_utc = ANALYSIS_START.tz_localize(LOCAL_TIMEZONE).tz_convert("UTC")
    study_end_utc = STUDY_END_EXCLUSIVE.tz_localize(LOCAL_TIMEZONE).tz_convert("UTC")

    # Defining sequential screening checks without deleting episode records
    episode_checks = {
        "All saved episodes, including context": pd.Series(
            True, index=episode_records.index
        ),
        "Onset within estimation period": (
            episode_records["onset_time"].ge(analysis_start_utc)
            & episode_records["onset_time"].lt(study_end_utc)
        ),
        "Weekday daytime": episode_records["broad_period"].eq("weekday_daytime"),
        "Selected main cause": episode_records["cause_eligibility"].eq("main"),
        "Observed alert end": (
            episode_records["is_right_censored"].eq(False)
            & episode_records["end_time"].notna()
        ),
        "Successfully measured": episode_records["measure_status"].eq("measured"),
        "Measurement window not truncated": (
            episode_records["window_truncated"].eq(False)
        ),
        "No recorded alert gap near episode": (
            episode_records["on_alert_gap_day"].eq(False)
        ),
    }

    # Counting episodes remaining after each successive check
    remaining = pd.Series(True, index=episode_records.index)
    screening_rows = []

    for label, condition in episode_checks.items():
        remaining &= condition.fillna(False)
        screening_rows.append(
            {"screening_step": label, "episodes_remaining": int(remaining.sum())}
        )

    episode_records["passes_episode_screen"] = remaining

    episode_screening = pd.DataFrame(screening_rows)

    return episode_records, episode_screening, analysis_start_utc, study_end_utc

In [ ]:
episode_records, episode_screening, analysis_start_utc, study_end_utc = screen_episodes(
)

display_table(episode_screening)

The episode screen leaves **2,605 episodes** from 9,373 saved records, including context episodes. Restricting onset to the estimation dates and weekday daytime removes many records before the cause and measurement checks.

These are candidate episodes, not 2,605 treated incidents in the final estimate. They still need sustained loss, usable clock hours, clean baselines and suitable comparisons. I will build those requirements next.

In [ ]:
def construct_treatment_hours(analysis_start_utc, episode_records, mta_manifest, research_panel, study_end_utc):
    """Reconstructing loss spells, preserving boundary flags, and mapping fully exposed hours."""
    # Loading attributed bins while retaining all episodes for later overlap checks
    attributed_bins = pd.read_parquet(input_paths["mta_attributed_bins"])

    bin_key = [
        "episode_key",
        "route_id",
        "direction",
        "station_complex_id",
        "bin_utc",
    ]
    loss_flags = ["lost", "partial_lost", "access_lost"]

    # Checking the identifiers, timestamps, and measured-loss flags
    assert attributed_bins[bin_key].notna().all().all()
    assert not attributed_bins.duplicated(bin_key).any()
    assert attributed_bins[loss_flags].notna().all().all()
    assert attributed_bins["episode_key"].isin(
        episode_records["episode_key"]
    ).all()

    # Selecting bins belonging to the preliminary candidate episodes
    candidate_episode_keys = episode_records.loc[
        episode_records["passes_episode_screen"], "episode_key"
    ]

    candidate_bins = attributed_bins.loc[
        attributed_bins["episode_key"].isin(candidate_episode_keys)
    ].copy()

    # Combining partial and complete route loss into one service-loss indicator
    candidate_bins["service_loss"] = (
        candidate_bins["lost"] | candidate_bins["partial_lost"]
    )

    # Checking that access loss is already contained within route loss
    assert (
        ~candidate_bins["access_lost"] | candidate_bins["lost"]
    ).all()

    # Summarizing the bins before reconstructing continuous spells

    # Ordering loss bins within each episode and service location
    spell_series = [
        "episode_key",
        "route_id",
        "direction",
        "station_complex_id",
    ]

    loss_bins = (
        candidate_bins.loc[candidate_bins["service_loss"]]
        .sort_values(spell_series + ["bin_utc"])
        .copy()
    )

    BIN_LENGTH = pd.Timedelta(15, unit="min")

    # Starting a new spell whenever consecutive loss timestamps are not adjacent
    time_since_previous = (
        loss_bins.groupby(spell_series, sort=False)["bin_utc"].diff()
    )

    loss_bins["spell_id"] = time_since_previous.ne(BIN_LENGTH).cumsum()

    # Summarizing continuous spells while preserving impact and overlap information
    loss_spells = (
        loss_bins.groupby(spell_series + ["spell_id"], as_index=False)
        .agg(
            spell_start_utc=("bin_utc", "min"),
            last_bin_utc=("bin_utc", "max"),
            loss_bins=("bin_utc", "size"),
            has_partial_loss=("partial_lost", "any"),
            has_route_loss=("lost", "any"),
            has_directional_access_loss=("access_lost", "any"),
            shared_bin_rows=("shared_with_other_episode", "sum"),
        )
    )

    loss_spells["spell_end_utc"] = loss_spells["last_bin_utc"] + BIN_LENGTH
    loss_spells["duration_minutes"] = loss_spells["loss_bins"] * 15
    loss_spells["is_long_spell"] = loss_spells["duration_minutes"].ge(60)

    # Validating that each reconstructed spell has no internal time gaps
    elapsed_minutes = (
        loss_spells["spell_end_utc"] - loss_spells["spell_start_utc"]
    ).dt.total_seconds() / 60

    assert elapsed_minutes.eq(loss_spells["duration_minutes"]).all()
    assert loss_spells["loss_bins"].sum() == len(loss_bins)

    # Counting spells and distinct episodes separately
    long_spells = loss_spells.loc[loss_spells["is_long_spell"]]

    # Attaching episode timing to long spells
    long_spell_review = long_spells.merge(
        episode_records[
            [
                "episode_key",
                "onset_time",
                "window_start",
                "window_end",
                "pre_alert_loss_starts_at_reference_end",
            ]
        ],
        on="episode_key",
        how="left",
        validate="many_to_one",
    )

    # Expressing measured starts in New York local time
    local_start = long_spell_review["spell_start_utc"].dt.tz_convert(
        LOCAL_TIMEZONE
    )

    long_spell_review["starts_weekday_06_to_20"] = (
        local_start.dt.dayofweek.lt(5)
        & local_start.dt.hour.ge(6)
        & local_start.dt.hour.lt(20)
    )

    # Flagging study boundaries and potentially incomplete measurement
    long_spell_review["within_estimation_dates"] = (
        long_spell_review["spell_start_utc"].ge(analysis_start_utc)
        & long_spell_review["spell_end_utc"].le(study_end_utc)
    )

    long_spell_review["reaches_measurement_end"] = (
        long_spell_review["spell_end_utc"]
        >= long_spell_review["window_end"]
    )

    long_spell_review["starts_before_alert"] = (
        long_spell_review["spell_start_utc"]
        < long_spell_review["onset_time"]
    )

    # Summarizing diagnostic flags without removing spells
    review_flags = [
        "starts_weekday_06_to_20",
        "within_estimation_dates",
        "reaches_measurement_end",
        "starts_before_alert",
    ]

    # Reading the reference-period length from the saved run configuration
    assert "reference_period" in mta_manifest["settings"]
    # Parsing the saved reference period with an explicit nanosecond unit
    reference_period = pd.to_timedelta(
        mta_manifest["settings"]["reference_period"]
    ).as_unit("ns")

    # Identifying the first bin eligible for attribution after the reference period
    first_attributable_bin = (
        long_spell_review["window_start"] + reference_period
    )

    long_spell_review["starts_at_attribution_boundary"] = (
        long_spell_review["spell_start_utc"].eq(first_attributable_bin)
    )

    # Measuring the timing difference: negative values mean loss precedes the alert
    long_spell_review["start_minus_alert_minutes"] = (
        long_spell_review["spell_start_utc"]
        - long_spell_review["onset_time"]
    ).dt.total_seconds() / 60

    # Comparing boundary patterns at both spell and episode levels
    boundary_flags = {
        "Starts at attribution boundary": (
            long_spell_review["starts_at_attribution_boundary"]
        ),
        "Reaches measurement end": (
            long_spell_review["reaches_measurement_end"]
        ),
        "Touches either boundary": (
            long_spell_review["starts_at_attribution_boundary"]
            | long_spell_review["reaches_measurement_end"]
        ),
    }

    boundary_summary = pd.DataFrame(
        [
            {
                "timing_issue": label,
                "spells": int(mask.sum()),
                "episodes": long_spell_review.loc[
                    mask, "episode_key"
                ].nunique(),
            }
            for label, mask in boundary_flags.items()
        ]
    )


    # Inspecting the distribution of measured starts relative to alert onset

    # Loading and validating the station-complex-to-zone mapping
    station_zone_bridge = pd.read_parquet(
        input_paths["mta_station_zone_bridge"]
    )

    bridge_key = ["station_complex_id", "LocationID"]

    assert station_zone_bridge[bridge_key].notna().all().all()
    assert not station_zone_bridge.duplicated(bridge_key).any()

    # Checking mapping coverage before expanding spells across zones
    assert long_spell_review["station_complex_id"].isin(
        station_zone_bridge["station_complex_id"]
    ).all(), "Some long-spell complexes have no taxi-zone mapping."

    # Mapping spells while preserving complexes linked to multiple zones
    zone_long_spells = long_spell_review.merge(
        station_zone_bridge,
        on="station_complex_id",
        how="left",
        validate="many_to_many",
    )

    # Restricting mapped records to the research panel's zones
    zone_long_spells = zone_long_spells.loc[
        zone_long_spells["LocationID"].isin(research_panel["LocationID"].unique())
    ].copy()

    zone_long_spells["touches_measurement_boundary"] = (
        zone_long_spells["starts_at_attribution_boundary"]
        | zone_long_spells["reaches_measurement_end"]
    )
    zone_long_spells["has_shared_bins"] = (
        zone_long_spells["shared_bin_rows"].gt(0)
    )

    # Summarizing each episode-zone once, regardless of its number of spells
    episode_zone_review = (
        zone_long_spells.groupby(
            ["episode_key", "LocationID", "zone_borough"], as_index=False
        )
        .agg(
            long_spells=("spell_id", "nunique"),
            any_boundary_spell=("touches_measurement_boundary", "any"),
            any_shared_spell=("has_shared_bins", "any"),
        )
    )

    borough_spell_coverage = (
        episode_zone_review.groupby("zone_borough")
        .agg(
            episodes=("episode_key", "nunique"),
            affected_zones=("LocationID", "nunique"),
            episode_zone_pairs=("episode_key", "size"),
            pairs_with_boundary_spells=("any_boundary_spell", "sum"),
            pairs_with_shared_spells=("any_shared_spell", "sum"),
        )
    )

    # Finding complete clock hours contained within each mapped loss spell
    full_hour_start = zone_long_spells["spell_start_utc"].dt.ceil("h")
    full_hour_end = zone_long_spells["spell_end_utc"].dt.floor("h")

    zone_long_spells["contained_full_hours"] = (
        (full_hour_end - full_hour_start)
        .dt.total_seconds()
        .div(3600)
        .clip(lower=0)
        .astype("int64")
    )

    zone_long_spells["has_full_hour"] = (
        zone_long_spells["contained_full_hours"].gt(0)
    )

    # Flagging full-hour spells without the two recorded timing/attribution concerns
    zone_long_spells["has_full_hour_without_review_flags"] = (
        zone_long_spells["has_full_hour"]
        & ~zone_long_spells["touches_measurement_boundary"]
        & ~zone_long_spells["has_shared_bins"]
    )

    # Counting each episode-zone once rather than summing overlapping spell hours
    episode_zone_hour_coverage = (
        zone_long_spells.groupby(
            ["episode_key", "LocationID", "zone_borough"],
            as_index=False,
        )
        .agg(
            any_full_hour=("has_full_hour", "any"),
            any_full_hour_without_review_flags=(
                "has_full_hour_without_review_flags", "any"
            ),
        )
    )

    full_hour_coverage = (
        episode_zone_hour_coverage.groupby("zone_borough")
        .agg(
            long_spell_pairs=("episode_key", "size"),
            pairs_with_full_hour=("any_full_hour", "sum"),
            pairs_with_full_hour_without_review_flags=(
                "any_full_hour_without_review_flags", "sum"
            ),
        )
    )

    # Selecting spells for the provisional full-hour sample
    spell_hour_source = zone_long_spells.loc[
        zone_long_spells["has_full_hour_without_review_flags"],
        [
            "episode_key",
            "LocationID",
            "spell_start_utc",
            "spell_end_utc",
        ],
    ].copy()

    # Including 45-minute boundary hours from spells with a full clock hour
    hour_records = []

    for row in spell_hour_source.itertuples(index=False):
        spell_start = row.spell_start_utc
        spell_end = row.spell_end_utc

        # Checking that the spell contains at least one full clock hour
        assert (
            spell_end.floor("h") - spell_start.ceil("h")
        ) >= pd.Timedelta(hours=1)

        candidate_hours = pd.date_range(
            start=spell_start.floor("h"),
            end=spell_end,
            freq="h",
            inclusive="left",
        )

        for hour in candidate_hours:
            # Measuring overlap with this spell rather than adding across spells
            overlap_start = max(hour, spell_start)
            overlap_end = min(
                hour + pd.Timedelta(hours=1),
                spell_end,
            )
            loss_minutes = (
                overlap_end - overlap_start
            ).total_seconds() / 60

            if loss_minutes >= 45:
                hour_records.append(
                    (row.episode_key, row.LocationID, hour)
                )

    provisional_treated_hours = pd.DataFrame(
        hour_records,
        columns=["episode_key", "LocationID", "hour_utc"],
    ).drop_duplicates()

    # Converting UTC hours to the panel's New York local clock labels
    provisional_treated_hours["hour_local"] = (
        provisional_treated_hours["hour_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
    )

    # Attaching quality and calendar information without selecting on pickup outcomes
    provisional_treated_hours = provisional_treated_hours.merge(
        research_panel[
            [
                "LocationID",
                "hour_local",
                "zone_borough",
                "passes_initial_quality",
                "is_holiday",
            ]
        ],
        on=["LocationID", "hour_local"],
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    assert provisional_treated_hours["_merge"].eq("both").all()
    provisional_treated_hours = provisional_treated_hours.drop(columns="_merge")

    # Checking the actual treated hour against the daytime research scope
    local_hour = provisional_treated_hours["hour_local"]

    provisional_treated_hours["within_daytime_scope"] = (
        local_hour.dt.dayofweek.lt(5)
        & local_hour.dt.hour.ge(6)
        & local_hour.dt.hour.lt(20)
        & ~provisional_treated_hours["is_holiday"]
    )

    provisional_treated_hours["passes_hour_screen"] = (
        provisional_treated_hours["passes_initial_quality"]
        & provisional_treated_hours["within_daytime_scope"]
    )

    return attributed_bins, station_zone_bridge, zone_long_spells, provisional_treated_hours

In [ ]:
# Rebuilding treatment hours using the revised 45-minute rule
attributed_bins, station_zone_bridge, zone_long_spells, provisional_treated_hours = (
    construct_treatment_hours(
        analysis_start_utc,
        episode_records,
        mta_manifest,
        research_panel,
        study_end_utc,
    )
)

# Reconstructing the original full-hour membership for comparison
full_hour_keys = set()

eligible_spells = zone_long_spells.loc[
    zone_long_spells["has_full_hour_without_review_flags"]
]

for row in eligible_spells.itertuples(index=False):
    for hour in pd.date_range(
        start=row.spell_start_utc.ceil("h"),
        end=row.spell_end_utc.floor("h"),
        freq="h",
        inclusive="left",
    ):
        full_hour_keys.add(
            (row.episode_key, row.LocationID, hour)
        )

# Identifying hours added by the revised rule
hour_keys = ["episode_key", "LocationID", "hour_utc"]
provisional_treated_hours["added_by_45_minute_rule"] = [
    key not in full_hour_keys
    for key in provisional_treated_hours[hour_keys].itertuples(
        index=False, name=None
    )
]

# Comparing hours that pass the existing quality and daytime screen
screened_hours = provisional_treated_hours.loc[
    provisional_treated_hours["passes_hour_screen"].eq(True)
]

hour_rule_comparison = (
    screened_hours.groupby("zone_borough")
    .agg(
        revised_hours=("hour_utc", "size"),
        added_hours=("added_by_45_minute_rule", "sum"),
    )
)

hour_rule_comparison.loc["Total"] = hour_rule_comparison.sum()
hour_rule_comparison["original_hours"] = (
    hour_rule_comparison["revised_hours"]
    - hour_rule_comparison["added_hours"]
)

display_table(
    hour_rule_comparison[
        ["original_hours", "added_hours", "revised_hours"]
    ],
    caption="Treatment-hour coverage before overlap, baseline and matching checks",
    labels={
        "original_hours": "Original full hours",
        "added_hours": "Added boundary hours",
        "revised_hours": "Revised total hours",
    },
)

The 45-minute rule adds **229 hours** to the original 2,201 hours after the initial quality and daytime screen, giving **2,430 hours**, an increase of 10.4%. This is before overlap, baseline and matching exclusions.

I now count each episode–zone–hour once, even if several station series identify it. Each pair also needs three clean baseline hours before the earlier warning/exposure anchor. Baseline hours can fall outside daytime; the daytime restriction applies to retained treatment hours and matching history.

In [ ]:
def prepare_event_candidates(attributed_bins, episode_records, provisional_treated_hours, research_panel, station_zone_bridge):
    """Constructing candidate pairs, pre-event baselines, and the eight-week history boundaries."""
    # Mapping all attributed flags to zone-hours, including noncandidate episodes
    all_episode_zone_hours = attributed_bins[
        ["episode_key", "station_complex_id", "bin_utc"]
    ].merge(
        station_zone_bridge[["station_complex_id", "LocationID"]],
        on="station_complex_id",
        how="inner",
        validate="many_to_many",
    )

    all_episode_zone_hours["hour_local"] = (
        all_episode_zone_hours["bin_utc"]
        .dt.floor("h")
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
    )

    # Counting each episode once per zone-hour, regardless of affected stations
    all_episode_zone_hours = all_episode_zone_hours[
        ["episode_key", "LocationID", "hour_local"]
    ].drop_duplicates()

    measured_episode_counts = (
        all_episode_zone_hours.groupby(["LocationID", "hour_local"])
        .size()
        .rename("measured_episodes_in_zone_hour")
        .reset_index()
    )

    # Reviewing only hours that passed the preceding screen
    treated_hour_overlap = provisional_treated_hours.loc[
        provisional_treated_hours["passes_hour_screen"]
    ].copy()

    treated_hour_overlap = treated_hour_overlap.merge(
        measured_episode_counts,
        on=["LocationID", "hour_local"],
        how="left",
        validate="many_to_one",
    )

    assert treated_hour_overlap[
        "measured_episodes_in_zone_hour"
    ].notna().all()
    assert treated_hour_overlap["measured_episodes_in_zone_hour"].ge(1).all()

    # Flagging additional episodes and recorded background loss
    treated_hour_overlap["other_measured_episode"] = (
        treated_hour_overlap["measured_episodes_in_zone_hour"].gt(1)
    )

    treated_hour_overlap = treated_hour_overlap.merge(
        research_panel[
            ["LocationID", "hour_local", "background_route_lost_minutes"]
        ],
        on=["LocationID", "hour_local"],
        how="left",
        validate="many_to_one",
    )

    treated_hour_overlap["has_background_loss"] = (
        treated_hour_overlap["background_route_lost_minutes"].gt(0)
    )

    treated_hour_overlap["passes_measured_overlap_screen"] = (
        ~treated_hour_overlap["other_measured_episode"]
        & ~treated_hour_overlap["has_background_loss"]
    )

    # Selecting episode-zone pairs with at least one surviving treatment hour
    candidate_pairs = (
        treated_hour_overlap.loc[
            treated_hour_overlap["passes_measured_overlap_screen"],
            ["episode_key", "LocationID", "zone_borough"],
        ]
        .drop_duplicates()
    )

    # Finding the earliest attributed exposure, including shorter losses and diversions
    first_zone_exposure = (
        attributed_bins[
            ["episode_key", "station_complex_id", "bin_utc"]
        ]
        .merge(
            station_zone_bridge[["station_complex_id", "LocationID"]],
            on="station_complex_id",
            how="inner",
            validate="many_to_many",
        )
        .groupby(["episode_key", "LocationID"], as_index=False)
        .agg(first_exposure_utc=("bin_utc", "min"))
    )

    pair_anchors = (
        candidate_pairs.merge(
            first_zone_exposure,
            on=["episode_key", "LocationID"],
            validate="one_to_one",
        )
        .merge(
            episode_records[["episode_key", "incident_start_uncapped"]],
            on="episode_key",
            validate="many_to_one",
        )
    )

    assert pair_anchors[
        ["first_exposure_utc", "incident_start_uncapped"]
    ].notna().all().all()

    pair_anchors["preperiod_anchor_utc"] = pair_anchors[
        ["first_exposure_utc", "incident_start_uncapped"]
    ].min(axis=1)

    # Creating three complete clock hours before the anchor
    preperiod_rows = []

    for row in pair_anchors.itertuples(index=False):
        hours = pd.date_range(
            end=row.preperiod_anchor_utc.floor("h"),
            periods=4,
            freq="h",
        )[:-1]

        preperiod_rows.extend(
            (row.episode_key, row.LocationID, row.zone_borough, hour)
            for hour in hours
        )

    preperiod_hours = pd.DataFrame(
        preperiod_rows,
        columns=["episode_key", "LocationID", "zone_borough", "hour_utc"],
    )

    preperiod_hours["hour_local"] = (
        preperiod_hours["hour_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
    )

    # Attaching panel quality and all-cause exposure information
    preperiod_hours = preperiod_hours.merge(
        research_panel[
            [
                "LocationID",
                "hour_local",
                "passes_initial_quality",
                "announced_minutes_all",
                "background_route_lost_minutes",
            ]
        ],
        on=["LocationID", "hour_local"],
        how="left",
        validate="many_to_one",
    )

    preperiod_hours = preperiod_hours.merge(
        measured_episode_counts,
        on=["LocationID", "hour_local"],
        how="left",
        validate="many_to_one",
    )

    # Treating missing panel coverage as unusable, rather than as clean exposure
    preperiod_hours["passes_preperiod_hour"] = (
        preperiod_hours["passes_initial_quality"].eq(True)
        & preperiod_hours["announced_minutes_all"].eq(0)
        & preperiod_hours["background_route_lost_minutes"].eq(0)
        & preperiod_hours["measured_episodes_in_zone_hour"].fillna(0).eq(0)
    )

    pair_preperiod_review = (
        preperiod_hours.groupby(
            ["episode_key", "LocationID", "zone_borough"], as_index=False
        )
        .agg(
            preperiod_hours=("hour_utc", "size"),
            passing_hours=("passes_preperiod_hour", "sum"),
        )
    )

    assert pair_preperiod_review["preperiod_hours"].eq(3).all()
    pair_preperiod_review["has_three_clean_pre_hours"] = (
        pair_preperiod_review["passing_hours"].eq(3)
    )

    # Retaining pairs with three usable pre-event hours
    usable_pair_keys = pair_preperiod_review.loc[
        pair_preperiod_review["has_three_clean_pre_hours"],
        ["episode_key", "LocationID"],
    ]

    matching_treated_hours = (
        treated_hour_overlap.loc[
            treated_hour_overlap["passes_measured_overlap_screen"]
        ]
        .merge(
            usable_pair_keys,
            on=["episode_key", "LocationID"],
            how="inner",
            validate="many_to_one",
        )
    )

    # Summarizing surviving treatment hours for each episode-zone
    matching_candidates = (
        matching_treated_hours.groupby(
            ["episode_key", "LocationID", "zone_borough"], as_index=False
        )
        .agg(
            treated_hours=("hour_utc", "size"),
            first_treated_hour_utc=("hour_utc", "min"),
            last_treated_hour_utc=("hour_utc", "max"),
        )
        .merge(
            pair_anchors[
                ["episode_key", "LocationID", "preperiod_anchor_utc"]
            ],
            on=["episode_key", "LocationID"],
            validate="one_to_one",
        )
        .merge(
            episode_records[["episode_key", "event_id"]],
            on="episode_key",
            validate="many_to_one",
        )
    )

    # Validating the candidate grain and retained pair count
    assert not matching_candidates.duplicated(
        ["episode_key", "LocationID"]
    ).any()
    assert len(matching_candidates) == len(usable_pair_keys)
    assert matching_candidates["treated_hours"].ge(1).all()

    # Comparing sample coverage across boroughs

    # Finding the beginning of each pair's three-hour pre-event window
    preperiod_starts = (
        preperiod_hours.groupby(
            ["episode_key", "LocationID"], as_index=False
        )
        .agg(preperiod_start_utc=("hour_utc", "min"))
    )

    matching_candidates = matching_candidates.merge(
        preperiod_starts,
        on=["episode_key", "LocationID"],
        validate="one_to_one",
    )

    # Ending matching history before the immediate pre-event window
    matching_candidates["history_end_local"] = (
        matching_candidates["preperiod_start_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
    )

    # Using eight local calendar weeks, with the end timestamp excluded
    matching_candidates["history_start_local"] = (
        matching_candidates["history_end_local"]
        - pd.DateOffset(weeks=MATCH_HISTORY_WEEKS)
    )

    # Checking calendar coverage before assessing usable historical observations
    matching_candidates["history_window_covered"] = (
        matching_candidates["history_start_local"].ge(PANEL_START)
        & matching_candidates["history_end_local"].le(STUDY_END_EXCLUSIVE)
    )

    assert (
        matching_candidates["history_start_local"]
        < matching_candidates["history_end_local"]
    ).all()

    # Defining the historical hours relevant to weekday daytime disruptions
    history_clock = research_panel["hour_local"]

    research_panel["within_history_scope"] = (
        history_clock.dt.dayofweek.lt(5)
        & history_clock.dt.hour.ge(6)
        & history_clock.dt.hour.lt(20)
        & research_panel["is_holiday"].eq(False)
    )

    # Checking that exposure fields are present before interpreting zeros
    history_exposure_cols = [
        "announced_minutes_all",
        "episodes_all",
        "background_route_lost_minutes",
    ]

    assert research_panel[history_exposure_cols].notna().all().all()

    # Flagging preliminary historical eligibility without removing observations
    research_panel["passes_preliminary_history_screen"] = (
        research_panel["within_history_scope"]
        & research_panel["passes_initial_quality"]
        & research_panel["announced_minutes_all"].eq(0)
        & research_panel["episodes_all"].eq(0)
        & research_panel["background_route_lost_minutes"].eq(0)
    )

    # Summarizing availability across the panel, before event-specific windows
    history_availability = (
        research_panel.groupby("zone_borough")
        .agg(
            weekday_daytime_hours=("within_history_scope", "sum"),
            preliminary_usable_hours=(
                "passes_preliminary_history_screen", "sum"
            ),
        )
    )

    history_availability["preliminary_usable_share"] = (
        history_availability["preliminary_usable_hours"]
        / history_availability["weekday_daytime_hours"]
    )

    return matching_candidates, matching_treated_hours, preperiod_hours, all_episode_zone_hours, measured_episode_counts, treated_hour_overlap, pair_preperiod_review

In [ ]:
matching_candidates, matching_treated_hours, preperiod_hours, all_episode_zone_hours, measured_episode_counts, treated_hour_overlap, pair_preperiod_review = prepare_event_candidates(
    attributed_bins,
    episode_records,
    provisional_treated_hours,
    research_panel,
    station_zone_bridge,
)

In [ ]:
# Reporting candidate coverage after applying all current quality rules
candidate_summary = matching_candidates.groupby('zone_borough').agg(
    Candidate_pairs=('episode_key', 'size'), Incidents=('event_id', 'nunique'),
    Treatment_hours=('treated_hours', 'sum'))
candidate_total = pd.DataFrame({'Candidate_pairs': [len(matching_candidates)],
    'Incidents': [matching_candidates['event_id'].nunique()],
    'Treatment_hours': [matching_candidates['treated_hours'].sum()]}, index=['Total'])
candidate_summary = pd.concat([candidate_summary, candidate_total])
candidate_summary['Share_of_pairs_pct'] = 100 * candidate_summary['Candidate_pairs'] / len(matching_candidates)
display_table(candidate_summary)

After other-disruption and baseline checks, **679 candidate pairs from 226 incidents contribute 947 treatment hours**. Brooklyn contributes 40.35% of pairs and Manhattan 31.22%, so borough representation is already uneven.

A candidate contributes about 1.39 treatment hours on average. This is mostly a short-window comparison, although some pairs contribute several hours. The earlier full-hour run had 666 candidates; the revised rule adds 13 candidates at this stage. I still need suitable comparison zones before estimating the contrast.

## 6. Selecting comparison zones

I want comparisons that resemble the treated zone before the event and have no recorded exposure during the event window. I exclude the treated zone itself, shared station complexes and zones with attributed flags from the same episode. Every hour from the first baseline hour through the last retained treatment hour must pass the comparison checks.

For historical matching, treated and comparison zones must contribute at identical timestamps in the preceding eight weeks. I require enough shared clean hours and comparable average volume, then rank eligible comparisons by pickup-count RMSE. Lower RMSE means closer historical counts; it does not prove the comparison is causally valid.

The expanded event windows leave at least one clean candidate comparison for 676 of 679 pairs. The historical matching requirements below are more restrictive. Scoring and selection stay separate so I can later exclude adjacent zones without inventing a new matching rule.

In [ ]:
def screen_comparison_zones(all_episode_zone_hours, matching_candidates, research_panel, station_zone_bridge):
    """Building the same-borough pool and screening recorded exposure over the event window."""
    # Listing the research zones and their boroughs
    research_zones = (
        research_panel[["LocationID", "zone_borough"]]
        .drop_duplicates()
        .rename(columns={"LocationID": "comparison_zone_id"})
    )

    assert research_zones["comparison_zone_id"].is_unique

    # Generating same-borough candidate comparisons
    comparison_pool = matching_candidates[
        ["episode_key", "LocationID", "zone_borough"]
    ].merge(
        research_zones,
        on="zone_borough",
        how="inner",
        validate="many_to_many",
    )

    comparison_pool = comparison_pool.loc[
        comparison_pool["LocationID"]
        != comparison_pool["comparison_zone_id"]
    ].copy()

    # Identifying zones that share a station complex
    shared_complex_pairs = (
        station_zone_bridge[["station_complex_id", "LocationID"]]
        .merge(
            station_zone_bridge[
                ["station_complex_id", "LocationID"]
            ].rename(columns={"LocationID": "comparison_zone_id"}),
            on="station_complex_id",
        )
        [["LocationID", "comparison_zone_id"]]
        .drop_duplicates()
        .assign(shares_station_complex=True)
    )

    comparison_pool = comparison_pool.merge(
        shared_complex_pairs,
        on=["LocationID", "comparison_zone_id"],
        how="left",
        validate="many_to_one",
    )

    # Identifying zones with any attributed flag from the same episode
    episode_footprint = (
        all_episode_zone_hours[["episode_key", "LocationID"]]
        .drop_duplicates()
        .rename(columns={"LocationID": "comparison_zone_id"})
        .assign(same_episode_exposure=True)
    )

    comparison_pool = comparison_pool.merge(
        episode_footprint,
        on=["episode_key", "comparison_zone_id"],
        how="left",
        validate="many_to_one",
    )

    comparison_pool["passes_initial_geographic_screen"] = (
        ~comparison_pool["shares_station_complex"].eq(True)
        & ~comparison_pool["same_episode_exposure"].eq(True)
    )

    # Counting potential comparisons for every treated episode-zone
    comparison_pool_counts = (
        comparison_pool.groupby(
            ["episode_key", "LocationID", "zone_borough"], as_index=False
        )
        .agg(
            potential_comparisons=(
                "passes_initial_geographic_screen", "sum"
            )
        )
    )

    # Creating the complete comparison window for each treated episode-zone
    window_rows = []

    for row in matching_candidates.itertuples(index=False):
        window_hours = pd.date_range(
            start=row.preperiod_start_utc,
            end=row.last_treated_hour_utc,
            freq="h",
        )

        window_rows.extend(
            (row.episode_key, row.LocationID, hour)
            for hour in window_hours
        )

    comparison_window_hours = pd.DataFrame(
        window_rows,
        columns=["episode_key", "LocationID", "hour_utc"],
    )

    comparison_window_hours["hour_local"] = (
        comparison_window_hours["hour_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
    )

    # Expanding geographically eligible comparisons across the required hours
    comparison_hour_checks = (
        comparison_pool.loc[
            comparison_pool["passes_initial_geographic_screen"],
            [
                "episode_key",
                "LocationID",
                "zone_borough",
                "comparison_zone_id",
            ],
        ]
        .merge(
            comparison_window_hours,
            on=["episode_key", "LocationID"],
            validate="many_to_many",
        )
    )

    # Attaching comparison-zone quality and exposure, without pickup outcomes
    comparison_quality = research_panel[
        [
            "LocationID",
            "hour_local",
            "passes_initial_quality",
            "announced_minutes_all",
            "episodes_all",
            "background_route_lost_minutes",
        ]
    ].rename(columns={"LocationID": "comparison_zone_id"})

    comparison_hour_checks = comparison_hour_checks.merge(
        comparison_quality,
        on=["comparison_zone_id", "hour_local"],
        how="left",
        validate="many_to_one",
    )

    # Requiring every hour to pass the recorded exposure checks
    comparison_hour_checks["passes_recorded_hour_checks"] = (
        comparison_hour_checks["passes_initial_quality"].eq(True)
        & comparison_hour_checks["announced_minutes_all"].eq(0)
        & comparison_hour_checks["episodes_all"].eq(0)
        & comparison_hour_checks["background_route_lost_minutes"].eq(0)
    )

    comparison_window_review = (
        comparison_hour_checks.groupby(
            [
                "episode_key",
                "LocationID",
                "zone_borough",
                "comparison_zone_id",
            ],
            as_index=False,
        )
        .agg(
            window_hours=("hour_utc", "size"),
            passes_recorded_window_checks=(
                "passes_recorded_hour_checks", "all"
            ),
        )
    )

    # Keeping zero-comparison cases visible in the summary
    comparison_window_counts = (
        comparison_window_review.groupby(
            ["episode_key", "LocationID", "zone_borough"],
            as_index=False,
        )
        .agg(
            comparisons_passing_window=(
                "passes_recorded_window_checks", "sum"
            )
        )
    )

    return comparison_window_review, comparison_window_hours, comparison_pool_counts

In [ ]:
# Screening comparisons across the revised event windows
comparison_window_review, comparison_window_hours, comparison_pool_counts = (
    screen_comparison_zones(
        all_episode_zone_hours,
        matching_candidates,
        research_panel,
        station_zone_bridge,
    )
)

# Counting comparisons that pass the complete window checks
passing_counts = (
    comparison_window_review.groupby(PAIR_KEYS)[
        "passes_recorded_window_checks"
    ]
    .sum()
    .rename("clean_comparisons")
    .reset_index()
)

# Keeping pairs with no eligible comparisons in the denominator
comparison_coverage = matching_candidates[
    PAIR_KEYS + ["zone_borough"]
].merge(
    passing_counts,
    on=PAIR_KEYS,
    how="left",
    validate="one_to_one",
)

comparison_coverage["clean_comparisons"] = (
    comparison_coverage["clean_comparisons"].fillna(0).astype(int)
)
comparison_coverage["has_comparison"] = (
    comparison_coverage["clean_comparisons"].gt(0)
)

comparison_screen_summary = (
    comparison_coverage.groupby("zone_borough")
    .agg(
        candidate_pairs=("episode_key", "size"),
        pairs_with_comparisons=("has_comparison", "sum"),
    )
)

comparison_screen_summary.loc["Total"] = (
    comparison_screen_summary.sum()
)
comparison_screen_summary["pairs_without_comparisons"] = (
    comparison_screen_summary["candidate_pairs"]
    - comparison_screen_summary["pairs_with_comparisons"]
)

display_table(
    comparison_screen_summary,
    caption="Candidate pairs with clean comparison zones",
)

In [ ]:
# Defining reusable diagnostics for pickups observed at identical timestamps
def summarize_pickup_comparison(shared_hours):
    required = ["hour_local", "treated_pickups", "comparison_pickups"]
    data = shared_hours[required].copy()

    assert not data.empty, "No shared historical observations."
    assert data[required].notna().all().all()
    assert not data["hour_local"].duplicated().any()
    assert data[["treated_pickups", "comparison_pickups"]].ge(0).all().all()

    outcomes = ["treated_pickups", "comparison_pickups"]
    means = data[outcomes].mean()

    assert means.gt(0).all(), "Both zones need positive mean demand."

    data["hour_of_day"] = data["hour_local"].dt.hour
    data["weekday"] = data["hour_local"].dt.dayofweek

    # Comparing profile shapes independently of overall pickup levels
    def profile_difference(group_column):
        profile = data.groupby(group_column)[outcomes].mean()
        shares = profile.div(profile.sum(axis=0), axis=1)

        difference_pp = 100 * (
            shares["treated_pickups"] - shares["comparison_pickups"]
        )
        return np.sqrt(np.mean(difference_pp**2))

    # Comparing timestamp-by-timestamp movements relative to each zone's mean
    normalized = data[outcomes].div(means, axis=1)
    tracking_difference = (
        normalized["treated_pickups"] - normalized["comparison_pickups"]
    )

    return pd.Series(
        {
            "shared_hours": len(data),
            "shared_dates": data["hour_local"].dt.date.nunique(),
            "hour_categories_observed": data["hour_of_day"].nunique(),
            "weekday_categories_observed": data["weekday"].nunique(),
            "treated_mean_pickups": means["treated_pickups"],
            "comparison_mean_pickups": means["comparison_pickups"],
            "comparison_to_treated_mean_ratio": (
                means["comparison_pickups"] / means["treated_pickups"]
            ),
            "hourly_profile_rmse_pp": profile_difference("hour_of_day"),
            "weekday_profile_rmse_pp": profile_difference("weekday"),
            "normalized_tracking_rmse": np.sqrt(
                np.mean(tracking_difference**2)
            ),
        },
        name="value",
    )

def score_pickup_match(shared_hours, eligible_history_hours):
    assert eligible_history_hours > 0
    assert len(shared_hours) <= eligible_history_hours

    result = summarize_pickup_comparison(shared_hours).to_dict()

    # Measuring disagreement on the original pickup-count scale
    differences = (
        shared_hours["treated_pickups"]
        - shared_hours["comparison_pickups"]
    )

    result["pickup_count_rmse"] = float(
        np.sqrt(np.mean(differences**2))
    )
    result["shared_history_share"] = (
        len(shared_hours) / eligible_history_hours
    )

    # Applying coverage and volume requirements before ranking
    result["passes_matching_requirements"] = bool(
        result["shared_hours"] >= MIN_SHARED_HISTORY_HOURS
        and result["shared_history_share"] >= MIN_SHARED_HISTORY_SHARE
        and MIN_VOLUME_RATIO
        <= result["comparison_to_treated_mean_ratio"]
        <= MAX_VOLUME_RATIO
    )

    return result

In [ ]:
def score_comparison_candidates(comparison_window_review, matching_candidates, research_panel):
    """Scoring every screened comparison on common historical observations."""
    # Indexing zone histories once to avoid repeatedly scanning the full panel
    zone_histories = {
        zone_id: group[
            [
                "hour_local",
                "total_pickups",
                "within_history_scope",
                "passes_preliminary_history_screen",
            ]
        ]
        .set_index("hour_local")
        .sort_index()
        for zone_id, group in research_panel.groupby("LocationID")
    }

    # Collecting comparisons that passed the recorded event-window checks
    comparison_lookup = (
        comparison_window_review.loc[
            comparison_window_review["passes_recorded_window_checks"]
        ]
        .groupby(["episode_key", "LocationID"])["comparison_zone_id"]
        .agg(list)
        .to_dict()
    )

    matching_score_rows = []

    # Scoring candidates using historical outcomes only
    for event in matching_candidates.itertuples(index=False):
        comparison_ids = comparison_lookup.get(
            (event.episode_key, event.LocationID), []
        )

        treated = zone_histories[event.LocationID].loc[
            event.history_start_local:event.history_end_local
        ]
        treated = treated.loc[
            (treated.index < event.history_end_local)
            & treated["within_history_scope"]
        ]

        eligible_hours = len(treated)

        treated_usable = treated.loc[
            treated["passes_preliminary_history_screen"],
            ["total_pickups"],
        ].rename(columns={"total_pickups": "treated_pickups"})

        for comparison_id in comparison_ids:
            comparison = zone_histories[comparison_id].loc[
                event.history_start_local:event.history_end_local
            ]
            comparison = comparison.loc[
                (comparison.index < event.history_end_local)
                & comparison["within_history_scope"]
                & comparison["passes_preliminary_history_screen"],
                ["total_pickups"],
            ].rename(columns={"total_pickups": "comparison_pickups"})

            shared = treated_usable.join(
                comparison,
                how="inner",
                validate="one_to_one",
            ).reset_index()

            result = {
                "episode_key": event.episode_key,
                "LocationID": event.LocationID,
                "zone_borough": event.zone_borough,
                "comparison_zone_id": comparison_id,
                "eligible_history_hours": eligible_hours,
                "shared_hours": len(shared),
                "passes_matching_requirements": False,
                "score_status": "insufficient_or_zero_demand_history",
            }

            if (
                eligible_hours > 0
                and not shared.empty
                and shared[["treated_pickups", "comparison_pickups"]]
                .mean().gt(0).all()
            ):
                result.update(score_pickup_match(shared, eligible_hours))
                result["score_status"] = "calculated"

            matching_score_rows.append(result)

    matching_scores = pd.DataFrame(matching_score_rows)
    assert not matching_scores.empty

    return matching_scores


def select_matches(matching_scores, excluded_comparisons=None):
    """Keeping up to three lowest-RMSE comparisons that meet the matching requirements."""
    eligible = matching_scores.loc[matching_scores["passes_matching_requirements"]].copy()

    # Optionally removing specific treated/comparison zone combinations
    if excluded_comparisons is not None:
        excluded = (excluded_comparisons[["LocationID", "comparison_zone_id"]]
                    .drop_duplicates().assign(excluded=True))
        eligible = eligible.merge(excluded, on=["LocationID", "comparison_zone_id"],
                                  how="left", validate="many_to_one")
        eligible = eligible.loc[eligible["excluded"].isna()].drop(columns="excluded")

    matches = eligible.sort_values(PAIR_KEYS + ["pickup_count_rmse", "comparison_zone_id"]).copy()
    matches["match_rank"] = matches.groupby(PAIR_KEYS).cumcount().add(1)
    return matches.loc[matches["match_rank"].le(MAX_COMPARISONS)].copy()


def summarize_matching(candidates, matches):
    """Reporting matched, fully matched and unmatched pairs by borough."""
    selected_counts = matches.groupby(PAIR_KEYS).size().rename("selected_comparisons").reset_index()
    coverage = candidates[PAIR_KEYS + ["zone_borough"]].merge(
        selected_counts, on=PAIR_KEYS, how="left", validate="one_to_one")
    coverage["selected_comparisons"] = coverage["selected_comparisons"].fillna(0).astype(int)

    summary = coverage.groupby("zone_borough").agg(
        candidate_pairs=("episode_key", "size"),
        pairs_with_matches=("selected_comparisons", lambda v: v.gt(0).sum()),
        pairs_with_three_matches=("selected_comparisons", lambda v: v.eq(3).sum()),
        pairs_without_matches=("selected_comparisons", lambda v: v.eq(0).sum()),
    )
    summary.loc["Total"] = summary.sum()
    for column, label in [("pairs_with_matches", "matched_pct"),
                          ("pairs_with_three_matches", "three_matches_pct"),
                          ("pairs_without_matches", "unmatched_pct")]:
        summary[label] = 100 * summary[column] / summary["candidate_pairs"]
    return coverage, summary

In [ ]:
matching_scores = score_comparison_candidates(
    comparison_window_review,
    matching_candidates,
    research_panel,
)
preliminary_matches = select_matches(matching_scores)
matching_coverage, matching_summary = summarize_matching(
    matching_candidates,
    preliminary_matches,
)

display_table(matching_summary)

In [ ]:
# Showing the matched and unmatched share of candidate pairs by borough
def plot_matching_retention(summary):
    data = summary.drop(index="Total").reset_index()
    fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    y = np.arange(len(data))
    ax.barh(y, data["pairs_with_matches"], color="#10707b", label="Matched")
    ax.barh(y, data["pairs_without_matches"], left=data["pairs_with_matches"],
            color="#D9DDE2", label="Unmatched")
    for i, row in data.iterrows():
        ax.text(row["candidate_pairs"] + 4, i,
                f"{row['pairs_with_matches']:.0f}/{row['candidate_pairs']:.0f} ({row['matched_pct']:.1f}%)",
                va="center", fontsize=9)
    ax.set_yticks(y, data["zone_borough"])
    ax.invert_yaxis()
    ax.set_xlim(0, data["candidate_pairs"].max() * 1.38)
    ax.set_xlabel("Episode–zone pairs"); ax.set_ylabel("")
    ax.set_title("How much of the candidate sample has a match?", loc="left", pad=12)
    ax.grid(axis="y", visible=False); ax.legend(loc="lower right")
    sns.despine(ax=ax)
    return fig


plot_matching_retention(matching_summary)
plt.show()

Historical matching retains **370 of 679 pairs, or 54.49%**. Of these, 252 have three comparisons. Brooklyn retains 66.06% of its candidates, Queens 63.87%, the Bronx 45.95% and Manhattan 37.26%.

Clean comparison availability was high, but historical similarity and coverage remove many candidates. The estimate therefore describes disruptions with usable matches. I will not interpret it as representative of every eligible service-loss episode.

### Who remains after matching

Matching changes the population I can describe. I compare matched and unmatched candidates using baseline pickup volume, treatment timing and retained treatment duration. I do not use disruption-hour pickup outcomes to decide which group looks preferable.

In [ ]:
def describe_matched_sample(coverage, candidates, preperiod_hours, panel):
    """Comparing matched and unmatched candidate pairs on pre-event characteristics."""
    baseline = (
        preperiod_hours[PAIR_KEYS + ["hour_local"]]
        .merge(panel[["LocationID", "hour_local", "total_pickups"]],
               on=["LocationID", "hour_local"], how="left", validate="many_to_one")
        .groupby(PAIR_KEYS)["total_pickups"].mean().rename("baseline_pickups").reset_index()
    )
    data = (coverage.merge(candidates[PAIR_KEYS + ["treated_hours", "first_treated_hour_utc"]],
                           on=PAIR_KEYS, validate="one_to_one")
            .merge(baseline, on=PAIR_KEYS, validate="one_to_one"))
    data["sample"] = np.where(data["selected_comparisons"].gt(0), "Matched", "Unmatched")
    data["first_treated_local_hour"] = to_local(data["first_treated_hour_utc"]).dt.hour

    def summarize(group):
        return pd.Series({
            "pairs": len(group),
            "median_baseline_pickups": group["baseline_pickups"].median(),
            "mean_baseline_pickups": group["baseline_pickups"].mean(),
            "median_first_treated_hour": group["first_treated_local_hour"].median(),
            "share_first_hour_before_10_pct": 100 * group["first_treated_local_hour"].lt(10).mean(),
            "mean_treatment_hours": group["treated_hours"].mean(),
        })

    by_borough = data.groupby(["zone_borough", "sample"]).apply(summarize, include_groups=False)
    total = data.groupby("sample").apply(summarize, include_groups=False)
    total.index = pd.MultiIndex.from_product([["Total"], total.index], names=["zone_borough", "sample"])
    return data, pd.concat([by_borough, total])


matched_sample_data, matched_sample_summary = describe_matched_sample(
    matching_coverage, matching_candidates, preperiod_hours, research_panel)
display_table(matched_sample_summary, decimals=1)

Matched pairs have a median baseline of **117.2 pickups/hour**, compared with 121.0 for unmatched pairs. Both groups have a median first treatment hour of 11 and about 1.4 treatment hours on average. These overall summaries look fairly similar, but borough-level retention is still quite different.

For example, Queens matched pairs have lower median baseline volume than unmatched candidates. So matching does more than reduce the sample size, it changes which locations I can study. I will retain the rules and limit the interpretation to the matched sample.

## 7. Estimating the pickup contrast

### Building the hourly matched sample

At each actual hour, I average the selected comparison zones and subtract that average from treated-zone pickups. I then compare each pair's mean gap during retained treatment hours with its mean gap over the three baseline hours.

The matched sample has **370 pairs from 164 incidents**, with **1,110 baseline observations and 514 treatment observations**. The baseline count is exactly three per pair. Relative to the earlier full-hour run, this is a net gain of 53 treatment observations. Each pair receives equal weight in the contrast, regardless of how many treatment hours it contributes.

In [ ]:
def build_did_sample(episode_records, matching_treated_hours, preliminary_matches, preperiod_hours, research_panel):
    """Building matched hourly outcomes and retaining weather for the same observations."""
    # Identifying matched episode-zone pairs
    pair_keys = ["episode_key", "LocationID"]

    matched_pairs = preliminary_matches[
        pair_keys + ["zone_borough"]
    ].drop_duplicates()

    # Combining the three pre-event hours with retained treatment hours
    did_hours = pd.concat(
        [
            preperiod_hours[
                pair_keys + ["hour_utc", "hour_local"]
            ].assign(period="before"),
            matching_treated_hours[
                pair_keys + ["hour_utc", "hour_local"]
            ].assign(period="during"),
        ],
        ignore_index=True,
    ).merge(
        matched_pairs,
        on=pair_keys,
        how="inner",
        validate="many_to_one",
    )

    assert not did_hours.duplicated(pair_keys + ["hour_utc"]).any()

    # Attaching treated-zone pickups and observed weather
    outcome_controls = [
        "total_pickups",
        "temperature_f",
        "precipitation_in",
        "rain_in",
        "snowfall_in",
        "wind_speed_mph",
    ]

    panel_values = research_panel[
        ["LocationID", "hour_local", *outcome_controls]
    ]

    did_event_hours = did_hours.merge(
        panel_values.rename(
            columns={name: f"treated_{name}" for name in outcome_controls}
        ),
        on=["LocationID", "hour_local"],
        how="left",
        validate="many_to_one",
    )

    # Attaching each selected comparison zone at the identical hours
    comparison_rows = did_hours.merge(
        preliminary_matches[pair_keys + ["comparison_zone_id"]],
        on=pair_keys,
        validate="many_to_many",
    ).merge(
        panel_values.rename(columns={"LocationID": "comparison_zone_id"}),
        on=["comparison_zone_id", "hour_local"],
        how="left",
        validate="many_to_one",
    )

    assert comparison_rows[outcome_controls].notna().all().all()

    # Averaging comparison-zone outcomes and weather with equal zone weights
    hour_keys = pair_keys + ["hour_utc"]

    comparison_means = (
        comparison_rows.groupby(hour_keys)[outcome_controls]
        .mean()
        .add_prefix("comparison_")
        .reset_index()
    )

    comparison_counts = (
        comparison_rows.groupby(hour_keys)["comparison_zone_id"]
        .nunique()
        .rename("comparison_count")
        .reset_index()
    )

    did_event_hours = (
        did_event_hours.merge(
            comparison_means, on=hour_keys, validate="one_to_one"
        )
        .merge(
            comparison_counts, on=hour_keys, validate="one_to_one"
        )
        .merge(
            episode_records[["episode_key", "event_id"]],
            on="episode_key",
            validate="many_to_one",
        )
    )

    # Validating complete matched outcomes and stable comparison membership
    assert did_event_hours[
        ["treated_total_pickups", "comparison_total_pickups"]
    ].notna().all().all()

    expected_comparison_counts = (
        preliminary_matches.groupby(pair_keys)
        .size()
        .rename("expected_comparison_count")
        .reset_index()
    )

    did_event_hours = did_event_hours.merge(
        expected_comparison_counts,
        on=pair_keys,
        validate="many_to_one",
    )

    assert did_event_hours["comparison_count"].eq(
        did_event_hours["expected_comparison_count"]
    ).all()

    # Calculating the hourly gap without estimating the disruption effect yet
    did_event_hours["pickup_gap"] = (
        did_event_hours["treated_total_pickups"]
        - did_event_hours["comparison_total_pickups"]
    )

    # Identifying episode-zone pairs for sample reporting
    did_summary_data = did_event_hours.assign(
        episode_zone_pair=(
            did_event_hours["episode_key"].astype(str)
            + ":"
            + did_event_hours["LocationID"].astype(str)
        )
    )

    summary_measures = {
        "episode_zone_hours": ("pickup_gap", "size"),
        "episode_zone_pairs": ("episode_zone_pair", "nunique"),
        "episodes": ("episode_key", "nunique"),
        "incidents": ("event_id", "nunique"),
    }

    # Summarizing each borough and period
    did_sample_summary = (
        did_summary_data.groupby(["zone_borough", "period"])
        .agg(**summary_measures)
        .reset_index()
    )

    # Calculating period totals directly, avoiding double-counted incidents
    did_period_totals = (
        did_summary_data.groupby("period")
        .agg(**summary_measures)
        .reset_index()
        .assign(zone_borough="Total")
    )

    # Reporting each borough's share of observations within that period
    period_hour_totals = did_period_totals.set_index("period")[
        "episode_zone_hours"
    ]

    did_sample_summary = pd.concat(
        [did_sample_summary, did_period_totals],
        ignore_index=True,
    )

    did_sample_summary["share_of_period_hours_pct"] = (
        100
        * did_sample_summary["episode_zone_hours"]
        / did_sample_summary["period"].map(period_hour_totals)
    )

    did_sample_summary = did_sample_summary.set_index(
        ["zone_borough", "period"]
    ).sort_index()

    return did_event_hours, matched_pairs, did_sample_summary

In [ ]:
did_event_hours, matched_pairs, did_sample_summary = build_did_sample(
    episode_records,
    matching_treated_hours,
    preliminary_matches,
    preperiod_hours,
    research_panel,
)

display_table(did_sample_summary)

In [ ]:
def calculate_pair_contrasts(did_event_hours):
    """Calculating equal-pair difference-in-differences contrasts on the pickup-count scale."""
    # Averaging outcomes separately within each episode-zone and period
    pair_period_means = (
        did_event_hours.groupby(
            ["episode_key", "LocationID", "zone_borough", "event_id", "period"]
        )[
            [
                "treated_total_pickups",
                "comparison_total_pickups",
                "pickup_gap",
            ]
        ]
        .mean()
    )

    # Aligning each pair's pre-event and treatment averages
    did_pair_estimates = pair_period_means.unstack("period")

    required_columns = [
        (outcome, period)
        for outcome in [
            "treated_total_pickups",
            "comparison_total_pickups",
            "pickup_gap",
        ]
        for period in ["before", "during"]
    ]

    assert did_pair_estimates[required_columns].notna().all().all()

    did_pair_estimates.columns = [
        f"{outcome}_{period}"
        for outcome, period in did_pair_estimates.columns
    ]
    did_pair_estimates = did_pair_estimates.reset_index()

    # Calculating changes in pickups per hour and their difference
    did_pair_estimates["treated_change"] = (
        did_pair_estimates["treated_total_pickups_during"]
        - did_pair_estimates["treated_total_pickups_before"]
    )

    did_pair_estimates["comparison_change"] = (
        did_pair_estimates["comparison_total_pickups_during"]
        - did_pair_estimates["comparison_total_pickups_before"]
    )

    did_pair_estimates["did_pickups_per_hour"] = (
        did_pair_estimates["treated_change"]
        - did_pair_estimates["comparison_change"]
    )

    # Verifying the equivalent change-in-gap calculation
    assert np.allclose(
        did_pair_estimates["did_pickups_per_hour"],
        did_pair_estimates["pickup_gap_during"]
        - did_pair_estimates["pickup_gap_before"],
    )

    # Summarizing with equal weight for each episode-zone pair
    summary_spec = {
        "episode_zone_pairs": ("episode_key", "size"),
        "incidents": ("event_id", "nunique"),
        "mean_treated_change": ("treated_change", "mean"),
        "mean_comparison_change": ("comparison_change", "mean"),
        "mean_did_pickups_per_hour": ("did_pickups_per_hour", "mean"),
        "median_did_pickups_per_hour": ("did_pickups_per_hour", "median"),
    }

    did_borough_summary = (
        did_pair_estimates.groupby("zone_borough").agg(**summary_spec)
    )

    did_total_summary = (
        did_pair_estimates.assign(zone_borough="Total")
        .groupby("zone_borough")
        .agg(**summary_spec)
    )

    did_borough_summary = pd.concat(
        [did_borough_summary, did_total_summary]
    )

    did_borough_summary["share_of_pairs_pct"] = (
        100
        * did_borough_summary["episode_zone_pairs"]
        / len(did_pair_estimates)
    )

    return did_pair_estimates, did_borough_summary

In [ ]:
did_event_hours_raw = did_event_hours  # kept for the sensitivity table
did_pair_estimates, did_borough_summary = calculate_pair_contrasts(did_event_hours)
display_table(did_borough_summary)

Treated-zone pickups rise by **45.35 per hour** on average, while comparison-zone pickups rise by **28.02**. Their difference gives the raw contrast of **17.33 pickups/hour**, compared with 16.92 in the previous full-hour run.

The comparison change is substantial. Looking only at the treated increase would overstate what this design attributes to disruption. I will keep the raw contrast as a reference and next account for the zones' usual weekday–hour patterns.

### Removing the usual weekday and hour gap

Baseline and treatment hours often fall at different clock times. If the treated zone normally gains more pickups than its comparisons around that time, part of the raw contrast could simply be its usual daily pattern.

For each pair, I estimate the usual gap by weekday and hour from its earlier clean eight-week history. All selected zones must be usable at the same historical timestamps. I subtract this usual gap from each event-hour gap, keeping the selected comparisons fixed.

First I check every required weekday–hour combination. Missing combinations lead to exclusion rather than filling an unsupported historical mean.

In [ ]:
# Checking historical support while keeping the selected matches fixed
def check_weekday_hour_support(panel, candidates, matches, event_hours):
    keys = ["episode_key", "LocationID"]

    assert not matches.duplicated(keys + ["comparison_zone_id"]).any()
    assert not candidates.duplicated(keys).any()

    # Keeping observed outcomes with known, clean all-cause exposure
    clean = (
        panel["passes_initial_quality"].eq(True)
        & panel["announced_minutes_all"].eq(0)
        & panel["episodes_all"].eq(0)
        & panel["background_route_lost_minutes"].eq(0)
        & panel["is_holiday"].eq(False)
        & panel["total_pickups"].notna()
    )

    # Including early hours needed by the three-hour baseline
    histories = {
        zone: group.set_index("hour_local")["total_pickups"].sort_index()
        for zone, group in panel.loc[
            clean, ["LocationID", "hour_local", "total_pickups"]
        ].groupby("LocationID")
    }

    required = event_hours[
        keys + ["zone_borough", "period", "hour_local"]
    ].copy()
    required["weekday"] = required["hour_local"].dt.dayofweek
    required["hour_of_day"] = required["hour_local"].dt.hour
    required = required.drop_duplicates(
        keys + ["period", "weekday", "hour_of_day"]
    )

    windows = candidates.set_index(keys)
    support_rows = []

    # Requiring identical historical timestamps across all selected zones
    for (episode, treated_zone), group in matches.groupby(keys):
        window = windows.loc[(episode, treated_zone)]
        zone_ids = [
            treated_zone,
            *group["comparison_zone_id"].tolist(),
        ]
        assert len(zone_ids) == len(set(zone_ids))

        shared = pd.concat(
            [histories[zone].rename(zone) for zone in zone_ids],
            axis=1,
            join="inner",
        ).dropna()

        shared = shared.loc[
            (shared.index >= window["history_start_local"])
            & (shared.index < window["history_end_local"])
        ]

        assert shared.index.is_unique
        assert (shared.index < window["history_end_local"]).all()

        counts = (
            pd.DataFrame({
                "weekday": shared.index.dayofweek,
                "hour_of_day": shared.index.hour,
                "historical_date": shared.index.date,
            })
            .groupby(["weekday", "hour_of_day"])["historical_date"]
            .nunique()
            .rename("clean_history_dates")
            .reset_index()
        )
        counts["episode_key"] = episode
        counts["LocationID"] = treated_zone
        support_rows.append(counts)

    # Retaining required combinations even when historical support is absent
    coverage = required.merge(
        pd.concat(support_rows, ignore_index=True),
        on=keys + ["weekday", "hour_of_day"],
        how="left",
        validate="many_to_one",
    )
    coverage["clean_history_dates"] = (
        coverage["clean_history_dates"].fillna(0).astype(int)
    )

    # Summarizing the weakest supported combination for each treated pair
    pair_support = (
        coverage.groupby(keys + ["zone_borough"], as_index=False)
        .agg(
            minimum_clean_dates=("clean_history_dates", "min"),
        )
    )
    assert len(pair_support) == len(event_hours[keys].drop_duplicates())

    def summarize(group):
        return pd.Series({
            "matched_pairs": len(group),
            "pairs_with_all_combinations": int(
                group["minimum_clean_dates"].ge(1).sum()
            ),
            "pairs_missing_combinations": int(
                group["minimum_clean_dates"].eq(0).sum()
            ),
            "minimum_dates_per_combination": group["minimum_clean_dates"].min(),
            "median_pair_minimum_dates": group["minimum_clean_dates"].median(),
        })

    summary = pd.DataFrame({
        borough: summarize(group)
        for borough, group in pair_support.groupby("zone_borough")
    }).T
    summary.loc["Total"] = summarize(pair_support)
    summary["fully_covered_pct"] = (
        100 * summary["pairs_with_all_combinations"]
        / summary["matched_pairs"]
    )

    return coverage, pair_support, summary

In [ ]:
weekday_hour_coverage, weekday_hour_pair_support, weekday_hour_summary = check_weekday_hour_support(
    research_panel, matching_candidates, preliminary_matches, did_event_hours)
display_table(weekday_hour_summary, decimals=1)

**368 of 370 pairs have all required combinations.** Two Brooklyn pairs lack support and are excluded, leaving 368 pairs and 163 incidents for the primary analysis.

Coverage does not mean the historical means are precise. The median pair's weakest combination has only three clean dates, and some combinations have just one. I will keep this adjustment, while recognizing that estimated historical patterns add uncertainty not fully represented by the reported intervals.

In [ ]:
# Adjusting pickup gaps using earlier clean weekday-hour patterns
def adjust_historical_gaps(
    panel, candidates, matches, event_hours, pair_estimates, coverage
):
    keys = ["episode_key", "LocationID"]
    clock_keys = ["weekday", "hour_of_day"]

    # Retaining pairs with historical support for every required combination
    supported = (
        coverage.groupby(keys)["clean_history_dates"]
        .min()
        .ge(1)
    )
    supported_keys = supported.loc[supported].reset_index()[keys]

    selected = matches.merge(
        supported_keys, on=keys, validate="many_to_one"
    )
    windows = candidates.set_index(keys)

    clean = (
        panel["passes_initial_quality"].eq(True)
        & panel["announced_minutes_all"].eq(0)
        & panel["episodes_all"].eq(0)
        & panel["background_route_lost_minutes"].eq(0)
        & panel["is_holiday"].eq(False)
        & panel["total_pickups"].notna()
    )
    histories = {
        zone: group.set_index("hour_local")["total_pickups"].sort_index()
        for zone, group in panel.loc[
            clean, ["LocationID", "hour_local", "total_pickups"]
        ].groupby("LocationID")
    }

    profiles = []

    # Estimating historical gaps on timestamps shared by all selected zones
    for (episode, treated_zone), group in selected.groupby(keys):
        window = windows.loc[(episode, treated_zone)]
        controls = group["comparison_zone_id"].tolist()
        zone_ids = [treated_zone, *controls]

        shared = pd.concat(
            [histories[z].rename(z) for z in zone_ids],
            axis=1,
            join="inner",
        ).dropna()
        shared = shared.loc[
            (shared.index >= window["history_start_local"])
            & (shared.index < window["history_end_local"])
        ]

        assert shared.index.is_unique
        assert (shared.index < window["history_end_local"]).all()

        historical = pd.DataFrame({
            "weekday": shared.index.dayofweek,
            "hour_of_day": shared.index.hour,
            "normal_gap": (
                shared[treated_zone] - shared[controls].mean(axis=1)
            ).to_numpy(),
        })

        profile = (
            historical.groupby(clock_keys, as_index=False)
            .agg(
                historical_gap=("normal_gap", "mean"),
                historical_dates=("normal_gap", "size"),
            )
        )
        profile["episode_key"] = episode
        profile["LocationID"] = treated_zone
        profiles.append(profile)

    historical_profiles = pd.concat(profiles, ignore_index=True)

    # Attaching expected gaps without using event outcomes to fit them
    adjusted_hours = event_hours.merge(
        supported_keys, on=keys, validate="many_to_one"
    ).copy()
    adjusted_hours["weekday"] = adjusted_hours["hour_local"].dt.dayofweek
    adjusted_hours["hour_of_day"] = adjusted_hours["hour_local"].dt.hour

    adjusted_hours = adjusted_hours.merge(
        historical_profiles,
        on=keys + clock_keys,
        how="left",
        validate="many_to_one",
    )
    assert adjusted_hours["historical_gap"].notna().all()

    adjusted_hours["adjusted_gap"] = (
        adjusted_hours["pickup_gap"] - adjusted_hours["historical_gap"]
    )

    # Preserving equal pair weights by averaging within each period first
    periods = (
        adjusted_hours.groupby(keys + ["period"])["adjusted_gap"]
        .mean()
        .unstack("period")
    )
    assert periods[["before", "during"]].notna().all().all()

    periods["historical_adjusted_contrast"] = (
        periods["during"] - periods["before"]
    )

    adjusted_pairs = pair_estimates.merge(
        periods[["historical_adjusted_contrast"]].reset_index(),
        on=keys,
        validate="one_to_one",
    )
    assert len(adjusted_pairs) == len(supported_keys)

    return historical_profiles, adjusted_hours, adjusted_pairs

In [ ]:
historical_gap_profiles, historical_adjusted_hours, historical_adjusted_pairs = adjust_historical_gaps(
    research_panel, matching_candidates, preliminary_matches, did_event_hours,
    did_pair_estimates, weekday_hour_coverage)

### How much of the pickup gap could be weather?

Suppose the affected zone goes from 20 more pickups than its comparison zones before the disruption to 50 more during it. If its usual weekday and hour gap goes from 15 to 25, I get $(50-25)-(20-15)=20$. If the fitted weather contribution accounts for another 3, I am left with 17. I made up these numbers to show what gets subtracted at each step.

In the actual model, $A_{it}$ is the pickup gap after subtracting the usual weekday and hour gap. I fit:

$$A_{it}=\alpha_i+\beta D_{it}+\mathbf W_{it}^{T}\boldsymbol\gamma+\varepsilon_{it}.$$

$alpha_i$ gives each pair its own baseline. $D_{it}$ switches from zero before the disruption to one during it, and $eta$ is the contrast I report. $mathbf W_{it}$ contains rain, snow, wind and temperature for the affected zone and the average of its comparison zones. I give each pair the same total weight before and during, even when one pair has more hours than another. I fit the weather terms and the disruption contrast together on the retained event windows.

I scale the weather variables so the coefficients are easier to read. For example, rain is in tenths of an inch and temperature is in tens of degrees Fahrenheit. The model assumes a straight-line relationship for each weather variable and shares those slopes across pairs, with separate slopes for affected and comparison zones. Sports and concert flags enter the same-zone comparison later.

Suppose there is a snowfall in early December and the previous eight weeks had no snow. I would have no examples for estimating a separate historical snow response for that pair. Here I use weather variation across the event windows. The estimate can still be weak if there are few snowy hours or snow mostly coincides with disruptions. Comparing zones at the same hour helps with shared weather, but I still need a credible comparison zone.


In [ ]:
def fit_weather_did(hours, gap_column, cluster="incident", borough_contrasts=False):
    """Fitting the weighted DiD regression with pair baselines and linear weather terms."""
    data = hours.copy()
    data["event_zone_id"] = data["episode_key"].astype(str) + ":" + data["LocationID"].astype(str)
    data["during"] = data["period"].eq("during").astype(int)
    data["gap_outcome"] = data[gap_column]

    # Giving each episode-zone one unit of weight in each period
    period_sizes = data.groupby(["event_zone_id", "period"])["gap_outcome"].transform("size")
    data["analysis_weight"] = 1.0 / period_sizes
    weather_terms = add_weather_terms(data)
    assert data[["gap_outcome", *weather_terms]].notna().all().all()

    if borough_contrasts:
        boroughs = sorted(data["zone_borough"].unique())
        contrast_terms = {b: f"during_borough_{i}" for i, b in enumerate(boroughs)}
        for borough, term in contrast_terms.items():
            data[term] = (data["zone_borough"].eq(borough) & data["during"].eq(1)).astype(int)
    else:
        contrast_terms = {"Total": "during"}

    formula = ("gap_outcome ~ C(event_zone_id) + "
               + " + ".join([*contrast_terms.values(), *weather_terms]))
    model = smf.wls(formula, data=data, weights=data["analysis_weight"], missing="raise").fit(
        cov_type="cluster",
        cov_kwds={"groups": cluster_groups(data, cluster),
                  "use_correction": True, "df_correction": True},
        use_t=True)
    assert model.model.rank == model.model.exog.shape[1], "Redundant predictors; inspect first."
    return data, weather_terms, contrast_terms, model


def model_row(label, model, data, term="during"):
    """Formatting one contrast from a fitted model for summary tables."""
    lower, upper = model.conf_int().loc[term]
    return {"specification": label,
            "episode_zone_pairs": data["event_zone_id"].nunique(),
            "incidents": data["event_id"].nunique(),
            "contrast_pickups_per_hour": model.params[term],
            "ci95_lower": lower, "ci95_upper": upper}


def pair_weather_adjusted_contrasts(data, weather_terms, model, pair_table):
    """Recovering each pair's contrast after removing the fitted weather terms."""
    weather_part = data[weather_terms].to_numpy() @ model.params[weather_terms].to_numpy()
    periods = (data.assign(weather_adjusted_gap=data["gap_outcome"] - weather_part)
               .groupby(PAIR_KEYS + ["period"])["weather_adjusted_gap"].mean()
               .unstack("period"))
    contrasts = (periods["during"] - periods["before"]).rename("primary_contrast").reset_index()
    result = pair_table.merge(contrasts, on=PAIR_KEYS, validate="one_to_one")
    # With equal pair-period weights, the regression contrast is the mean of pair contrasts
    assert np.isclose(result["primary_contrast"].mean(), model.params["during"])
    return result

In [ ]:
# Primary specification: weekday-hour adjusted gap, linear weather, common contrast
primary_data, primary_weather_terms, _, primary_model = fit_weather_did(
    historical_adjusted_hours, "adjusted_gap")
primary_weather_beta = primary_model.params[primary_weather_terms]

primary_pairs = pair_weather_adjusted_contrasts(
    primary_data, primary_weather_terms, primary_model, historical_adjusted_pairs)

# The same model with two-way clustering by incident and treated zone
_, _, _, primary_model_two_way = fit_weather_did(
    historical_adjusted_hours, "adjusted_gap", cluster="two_way")

primary_table = pd.DataFrame([
    model_row("Primary, incident-clustered", primary_model, primary_data),
    model_row("Primary, two-way clustered", primary_model_two_way, primary_data),
]).set_index("specification")
display_table(primary_table, caption="Primary estimate: pickups per hour during sustained loss")

The primary contrast is **18.06 completed pickups per hour**, with an incident-clustered 95% interval of **12.06 to 24.05**. This is the average adjusted change for the retained episode–zone pairs, not an additional 18 pickups in every affected zone.

Two-way clustering by incident and treated zone gives the same estimate and an interval of 12.33 to 23.78. It is slightly narrower here; adding a clustering dimension does not guarantee a wider interval. Neither approach captures every source of uncertainty, including historical adjustment, exposure measurement and comparison selection.

I will keep the incident-clustered regression as the main result. The previous full-hour primary estimate was 18.08, so including the qualifying 45-minute hours barely changes the overall estimate.

I cluster uncertainty by incident because several affected zones can belong to the same disruption. The regression interval accounts for estimation within this fitted model. It takes the historical adjustment and matches as given. The pair-summary intervals also take the fitted weather coefficients as given, so I use the regression interval for the main result.


### Looking across boroughs

The overall contrast may hide differences in the retained borough samples. I allow the during-disruption coefficient to vary by borough, while keeping pair baselines and shared weather responses in the regression.

I also show borough averages of the recovered pair-level primary contrasts. Those summary intervals treat the fitted weather coefficients as fixed, so I will use the borough regression intervals for the main borough interpretation. Separate intervals do not formally test differences between boroughs.

In [ ]:
def summarize_borough_model(data, contrast_terms, model):
    """Extracting borough contrasts and their pair-share-weighted overall contrast."""
    pairs = data.drop_duplicates("event_zone_id")
    pair_counts = pairs["zone_borough"].value_counts()
    ci = model.conf_int()
    rows = [{"zone_borough": b,
             "episode_zone_pairs": int(pair_counts[b]),
             "incidents": pairs.loc[pairs["zone_borough"].eq(b), "event_id"].nunique(),
             "contrast_pickups_per_hour": model.params[t],
             "ci95_lower": ci.loc[t, 0], "ci95_upper": ci.loc[t, 1]}
            for b, t in contrast_terms.items()]

    weights = pd.Series(0.0, index=model.params.index)
    for borough, term in contrast_terms.items():
        weights[term] = pair_counts[borough] / len(pairs)
    overall = model.t_test(weights.to_numpy())
    lower, upper = overall.conf_int().ravel()
    rows.append({"zone_borough": "Total", "episode_zone_pairs": len(pairs),
                 "incidents": pairs["event_id"].nunique(),
                 "contrast_pickups_per_hour": np.asarray(overall.effect).item(),
                 "ci95_lower": lower, "ci95_upper": upper})
    return pd.DataFrame(rows).set_index("zone_borough")


borough_data, _, borough_terms, borough_model = fit_weather_did(
    historical_adjusted_hours, "adjusted_gap", borough_contrasts=True)
borough_summary = summarize_borough_model(borough_data, borough_terms, borough_model)
display_table(borough_summary, caption="Borough-varying contrasts (primary adjustments)")

primary_pair_borough_summary = clustered_mean_summary(primary_pairs, "primary_contrast", "estimate")
display_table(primary_pair_borough_summary,
              caption="Pair-level primary contrasts by borough (shared weather coefficients)")

Brooklyn has an estimated contrast of **22.11 pickups/hour [11.98, 32.24]**, and Queens **19.73 [7.21, 32.24]**. The Bronx estimate is **12.42 [2.81, 22.03]**. Manhattan is **10.21 [−1.19, 21.62]**, so its regression interval includes zero.

The pair-summary table gives Manhattan a lower bound just above zero, but it holds the fitted weather coefficients fixed. I would not call Manhattan's increase robust based on that table. The borough-varying overall contrast of 18.17 is also a different specification from the primary 18.06.

The adjustments matter more for borough detail than the overall total. Queens moves from a raw 31.32 to 19.73, while the Bronx moves from 5.15 to 12.42. I will use the later same-zone comparison to see which borough findings are most consistent.

## 8. Checking the timing and placebo dates

### Looking around the warning and exposure anchor

The primary estimate averages baseline and treatment hours. Here I keep hours separate, from −6 to +3 relative to the earlier warning/exposure anchor rounded down to the hour. I subtract each pair's adjusted gap at hour −1, which makes that reference zero by construction.

Before the anchor, I want to see whether treated and comparison zones were already moving apart. The same 201 pairs appear at every hour in the balanced curve. The unbalanced curve includes every usable pair at each hour, so its membership changes.

Hours 0 onward are after the anchor hour begins, not necessarily after measured loss begins. They need not meet the primary treatment threshold and may extend beyond retained loss. This is a timing diagnostic, not a recovery analysis.

In [ ]:
def build_event_study_panel(candidates, pair_table, matches, panel, measured_counts,
                            profiles, weather_beta, preperiod_hours):
    """Computing the adjusted pickup gap at each hour relative to the baseline anchor."""
    pairs = pair_table[PAIR_KEYS].drop_duplicates()
    base = candidates.merge(pairs, on=PAIR_KEYS, validate="one_to_one")

    # One row per pair and relative hour
    window = pd.DataFrame(
        [(r.episode_key, r.LocationID, r.zone_borough, r.event_id, k,
          r.preperiod_anchor_utc.floor("h") + pd.Timedelta(hours=k))
         for r in base.itertuples(index=False) for k in EVENT_STUDY_HOURS],
        columns=PAIR_KEYS + ["zone_borough", "event_id", "rel_hour", "hour_utc"])
    window["hour_local"] = to_local(window["hour_utc"])
    assert not window.duplicated(PAIR_KEYS + ["rel_hour"]).any()

    # Checking that hours -3 to -1 are exactly the baseline hours used in the estimate
    baseline_check = (
        window.loc[window["rel_hour"].between(-3, -1), PAIR_KEYS + ["hour_utc"]]
        .merge(preperiod_hours.merge(pairs, on=PAIR_KEYS)[PAIR_KEYS + ["hour_utc"]],
               on=PAIR_KEYS + ["hour_utc"], how="outer", indicator=True))
    assert baseline_check["_merge"].eq("both").all(), "Relative hours do not match the baseline."

    # Listing the treated zone and its selected comparisons
    members = pd.concat([
        pairs.assign(zone_id=pairs["LocationID"], role="treated"),
        matches.merge(pairs, on=PAIR_KEYS)[PAIR_KEYS + ["comparison_zone_id"]]
        .rename(columns={"comparison_zone_id": "zone_id"}).assign(role="comparison"),
    ], ignore_index=True)

    value_cols = ["total_pickups", *WEATHER_SCALES]
    panel_cols = ["LocationID", "hour_local", "passes_initial_quality", "announced_minutes_all",
                  "episodes_all", "background_route_lost_minutes", *value_cols]
    zone_hours = (
        window.merge(members, on=PAIR_KEYS, validate="many_to_many")
        .merge(panel[panel_cols].rename(columns={"LocationID": "zone_id"}),
               on=["zone_id", "hour_local"], how="left", validate="many_to_one")
        .merge(measured_counts.rename(columns={"LocationID": "zone_id"}),
               on=["zone_id", "hour_local"], how="left", validate="many_to_one"))
    zone_hours["measured_episodes_in_zone_hour"] = zone_hours["measured_episodes_in_zone_hour"].fillna(0)

    # Applying the baseline rule before the anchor and the quality rule afterwards
    quality = zone_hours["passes_initial_quality"].eq(True) & zone_hours["total_pickups"].notna()
    no_exposure = (zone_hours["announced_minutes_all"].eq(0)
                   & zone_hours["background_route_lost_minutes"].eq(0))
    treated_pre = quality & no_exposure & zone_hours["measured_episodes_in_zone_hour"].eq(0)
    comparison_any = quality & no_exposure & zone_hours["episodes_all"].eq(0)
    zone_hours["usable"] = np.select(
        [zone_hours["role"].eq("comparison"), zone_hours["rel_hour"].lt(0)],
        [comparison_any, treated_pre], default=quality)

    hour_keys = PAIR_KEYS + ["rel_hour"]
    treated = (zone_hours.loc[zone_hours["role"].eq("treated")]
               .set_index(hour_keys)[value_cols].add_prefix("treated_"))
    comparison = (zone_hours.loc[zone_hours["role"].eq("comparison")]
                  .groupby(hour_keys)[value_cols].mean().add_prefix("comparison_"))
    hour_usable = zone_hours.groupby(hour_keys)["usable"].all().rename("hour_usable")

    es = window.set_index(hour_keys).join(treated).join(comparison).join(hour_usable).reset_index()
    es["pickup_gap"] = es["treated_total_pickups"] - es["comparison_total_pickups"]
    es["weekday"] = es["hour_local"].dt.dayofweek
    es["hour_of_day"] = es["hour_local"].dt.hour
    es = es.merge(profiles[PAIR_KEYS + ["weekday", "hour_of_day", "historical_gap"]],
                  on=PAIR_KEYS + ["weekday", "hour_of_day"], how="left", validate="many_to_one")

    # Removing the usual weekday-hour gap and the primary model's weather terms
    terms = add_weather_terms(es)
    weather_part = es[terms].to_numpy() @ weather_beta.reindex(terms).to_numpy()
    es["adjusted_gap"] = es["pickup_gap"] - es["historical_gap"] - weather_part
    es["usable"] = es["hour_usable"] & es["adjusted_gap"].notna()

    # Normalizing to each pair's own hour -1
    reference = (es.loc[es["rel_hour"].eq(-1) & es["usable"]]
                 .set_index(PAIR_KEYS)["adjusted_gap"].rename("reference_gap").reset_index())
    es = es.merge(reference, on=PAIR_KEYS, how="left", validate="many_to_one")
    es["relative_gap"] = es["adjusted_gap"] - es["reference_gap"]
    es["usable"] = es["usable"] & es["reference_gap"].notna()

    balanced = es.groupby(PAIR_KEYS)["usable"].all().rename("balanced").reset_index()
    return es.merge(balanced, on=PAIR_KEYS, validate="many_to_one")


def estimate_event_study(es, balanced_only):
    """Averaging the normalized gap at each relative hour with incident clustering."""
    data = es.loc[es["usable"] & (es["balanced"] if balanced_only else True)]
    rows = []
    for k in EVENT_STUDY_HOURS:
        sample = data.loc[data["rel_hour"].eq(k)]
        if k == -1:
            rows.append({"rel_hour": k, "episode_zone_pairs": len(sample), "estimate": 0.0,
                         "ci95_lower": np.nan, "ci95_upper": np.nan})
            continue
        total = clustered_mean_summary(sample, "relative_gap", by_borough=False).loc["Total"]
        rows.append({"rel_hour": k, "episode_zone_pairs": int(total["episode_zone_pairs"]),
                     "estimate": total["estimate"],
                     "ci95_lower": total["ci95_lower"], "ci95_upper": total["ci95_upper"]})
    return pd.DataFrame(rows).set_index("rel_hour")


event_study_panel = build_event_study_panel(
    matching_candidates, primary_pairs, preliminary_matches, research_panel,
    measured_episode_counts, historical_gap_profiles, primary_weather_beta, preperiod_hours)

event_study_coverage = event_study_panel.groupby("rel_hour").agg(
    pairs=("usable", "size"), usable_pairs=("usable", "sum"))
event_study_coverage["usable_pct"] = 100 * event_study_coverage["usable_pairs"] / event_study_coverage["pairs"]
event_study_coverage["balanced_pairs"] = (
    event_study_panel.drop_duplicates(PAIR_KEYS)["balanced"].sum())
display_table(event_study_coverage, decimals=1, caption="Event-study coverage by relative hour")

event_study_balanced = estimate_event_study(event_study_panel, balanced_only=True)
event_study_unbalanced = estimate_event_study(event_study_panel, balanced_only=False)
display_table(pd.concat({"Balanced": event_study_balanced, "Unbalanced": event_study_unbalanced},
                        names=["sample"]),
              caption="Change in the adjusted gap since hour −1 (pickups per hour)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
for label, table, offset, color, marker in [
        ("Balanced", event_study_balanced, -0.08, "#10707B", "o"),
        ("Unbalanced", event_study_unbalanced, 0.08, "#CCAA66", "s")]:
    x = np.array(table.index) + offset
    est = table["estimate"].to_numpy()
    err = np.vstack([est - table["ci95_lower"].to_numpy(), table["ci95_upper"].to_numpy() - est])
    ax.errorbar(x, est, yerr=np.nan_to_num(err), fmt=marker, color=color, capsize=3,
                label=f"{label} ({int(table['episode_zone_pairs'].min())}+ pairs)")
ax.axhline(0, color="#555555", linestyle="--", linewidth=1)
ax.axvspan(-0.5, max(EVENT_STUDY_HOURS) + 0.5, color="#EDF4F5", zorder=0, label="Post-onset hours")
ax.set_xticks(EVENT_STUDY_HOURS)
ax.set_xlabel("Hours relative to the baseline anchor hour")
ax.set_ylabel("Change in adjusted gap since hour −1")
ax.set_title("Were treated zones already diverging before the disruption?", loc="left", pad=12)
ax.legend(loc="upper left", fontsize=9)
sns.despine(ax=ax)
plt.show()

The balanced curve uses **201 of 368 pairs, or 54.6%**. Its pre-anchor estimates range from −0.33 to 2.92 pickups/hour, and every reported pre-anchor interval includes zero. I do not see clear prior divergence in these estimates, although that is not proof of parallel trends and no joint pretrend test is implemented.

At hour +1, the balanced estimate reaches **20.26 [11.44, 29.09]** and the unbalanced estimate is **17.83 [11.64, 24.03]**. Both curves then decline. The similar direction is useful, but the balanced sample is selective and the unbalanced sample changes across hours.

I will retain this as supporting timing evidence. The decline by +3 does not establish service recovery, and the curve is not the same estimand as the average during-loss coefficient.

### Comparing other clean dates

I next ask whether the same zones produce similar contrasts on dates with no recorded exposure or quality problems. I shift the whole comparison window by complete weeks, preserving weekday and clock hour, and keep the selected zones, historical profiles and fitted weather coefficients fixed.

Eligible shifts are 9–12 weeks earlier or 1–12 weeks later. I exclude the earlier eight-week matching window because those observations helped select the comparisons. A shifted window must pass the clean checks for every treated and comparison hour and remain within usable panel coverage.

I compare real and placebo contrasts on the same covered pairs. Drawing one eligible placebo date per pair also gives a descriptive distribution of placebo means. Because future dates are allowed and fitted adjustments are reused, this is a retrospective diagnostic, not a prospective forecast evaluation or a new causal estimator.

In [ ]:
def build_multidate_placebo(window_hours, adjusted_hours, pair_table, matches, candidates,
                            panel, weather_beta):
    """Recomputing the primary contrast on clean whole-week-shifted windows."""
    pairs = pair_table[PAIR_KEYS].drop_duplicates()
    shifts = [*range(-PLACEBO_MAX_SHIFT_WEEKS, -MATCH_HISTORY_WEEKS), *range(1, PLACEBO_MAX_SHIFT_WEEKS + 1)]

    members = pd.concat([
        pairs.assign(zone_id=pairs["LocationID"], role="treated"),
        matches.merge(pairs, on=PAIR_KEYS)[PAIR_KEYS + ["comparison_zone_id"]]
        .rename(columns={"comparison_zone_id": "zone_id"}).assign(role="comparison"),
    ], ignore_index=True)

    # Shifting every hour of each pair's full comparison window
    windows = window_hours.merge(pairs, on=PAIR_KEYS)[PAIR_KEYS + ["hour_local"]]
    shifted = windows.merge(pd.DataFrame({"shift_weeks": shifts}), how="cross")
    shifted["placebo_hour_local"] = shifted["hour_local"] + pd.to_timedelta(7 * shifted["shift_weeks"], unit="D")

    # Rejecting shifts overlapping the matching history or the real event window
    bounds = (windows.groupby(PAIR_KEYS)["hour_local"].agg(event_start="min", event_end="max").reset_index()
              .merge(candidates[PAIR_KEYS + ["history_start_local", "history_end_local"]], on=PAIR_KEYS))
    shifted = shifted.merge(bounds, on=PAIR_KEYS, validate="many_to_one")
    shifted["overlaps"] = (
        shifted["placebo_hour_local"].between(shifted["history_start_local"], shifted["history_end_local"])
        | shifted["placebo_hour_local"].between(shifted["event_start"], shifted["event_end"]))

    quality_cols = ["passes_initial_quality", "announced_minutes_all", "episodes_all",
                    "background_route_lost_minutes", "is_holiday"]
    checks = (shifted.merge(members, on=PAIR_KEYS, validate="many_to_many")
              .merge(panel[["LocationID", "hour_local", *quality_cols]]
                     .rename(columns={"LocationID": "zone_id", "hour_local": "placebo_hour_local"}),
                     on=["zone_id", "placebo_hour_local"], how="left", validate="many_to_one"))
    checks["passes"] = (checks["passes_initial_quality"].eq(True)
                        & checks["announced_minutes_all"].eq(0) & checks["episodes_all"].eq(0)
                        & checks["background_route_lost_minutes"].eq(0)
                        & checks["is_holiday"].eq(False) & ~checks["overlaps"])
    eligible = (checks.groupby(PAIR_KEYS + ["shift_weeks"])["passes"].all()
                .loc[lambda s: s].reset_index()[PAIR_KEYS + ["shift_weeks"]])

    # Rebuilding the adjusted gap at the shifted hours
    hours = adjusted_hours[PAIR_KEYS + ["period", "hour_local", "historical_gap"]].merge(eligible, on=PAIR_KEYS)
    hours["placebo_hour_local"] = hours["hour_local"] + pd.to_timedelta(7 * hours["shift_weeks"], unit="D")
    assert (hours["placebo_hour_local"].dt.dayofweek == hours["hour_local"].dt.dayofweek).all()
    assert (hours["placebo_hour_local"].dt.hour == hours["hour_local"].dt.hour).all()

    value_cols = ["total_pickups", *WEATHER_SCALES]
    hour_keys = PAIR_KEYS + ["shift_weeks", "hour_local"]
    values = (hours.merge(members, on=PAIR_KEYS, validate="many_to_many")
              .merge(panel[["LocationID", "hour_local", *value_cols]]
                     .rename(columns={"LocationID": "zone_id", "hour_local": "placebo_hour_local"}),
                     on=["zone_id", "placebo_hour_local"], how="left", validate="many_to_one"))
    treated = values.loc[values["role"].eq("treated")].set_index(hour_keys)[value_cols].add_prefix("treated_")
    comparison = (values.loc[values["role"].eq("comparison")]
                  .groupby(hour_keys)[value_cols].mean().add_prefix("comparison_"))
    hours = hours.set_index(hour_keys).join(treated).join(comparison).reset_index()
    assert hours[["treated_total_pickups", "comparison_total_pickups"]].notna().all().all()

    terms = add_weather_terms(hours)
    weather_part = hours[terms].to_numpy() @ weather_beta.reindex(terms).to_numpy()
    hours["adjusted_gap"] = (hours["treated_total_pickups"] - hours["comparison_total_pickups"]
                             - hours["historical_gap"] - weather_part)

    periods = hours.groupby(PAIR_KEYS + ["shift_weeks", "period"])["adjusted_gap"].mean().unstack("period")
    assert periods[["before", "during"]].notna().all().all()
    contrasts = (periods["during"] - periods["before"]).rename("placebo_contrast").reset_index()
    return contrasts


placebo_contrasts = build_multidate_placebo(
    comparison_window_hours, historical_adjusted_hours, primary_pairs, preliminary_matches,
    matching_candidates, research_panel, primary_weather_beta)

# Summarizing placebo dates per pair and coverage by borough
pair_placebo = (placebo_contrasts.groupby(PAIR_KEYS)
                .agg(placebo_mean=("placebo_contrast", "mean"), placebo_dates=("placebo_contrast", "size"))
                .reset_index())
placebo_pairs = primary_pairs.merge(pair_placebo, on=PAIR_KEYS, how="inner", validate="one_to_one")
placebo_pairs["real_minus_placebo"] = placebo_pairs["primary_contrast"] - placebo_pairs["placebo_mean"]

placebo_coverage = primary_pairs.merge(pair_placebo, on=PAIR_KEYS, how="left").assign(
    covered=lambda d: d["placebo_dates"].notna(),
    placebo_dates=lambda d: d["placebo_dates"].fillna(0))
placebo_coverage_summary = placebo_coverage.groupby("zone_borough").agg(
    pairs=("covered", "size"), pairs_with_placebo=("covered", "sum"),
    median_placebo_dates=("placebo_dates", "median"))
placebo_coverage_summary.loc["Total"] = [len(placebo_coverage), placebo_coverage["covered"].sum(),
                                         placebo_coverage["placebo_dates"].median()]
placebo_coverage_summary["covered_pct"] = (
    100 * placebo_coverage_summary["pairs_with_placebo"] / placebo_coverage_summary["pairs"])
display_table(placebo_coverage_summary, decimals=1, caption="Placebo coverage")

placebo_summary = pd.concat({
    "Real event (same pairs)": clustered_mean_summary(placebo_pairs, "primary_contrast"),
    "Placebo mean": clustered_mean_summary(placebo_pairs, "placebo_mean"),
    "Real minus placebo": clustered_mean_summary(placebo_pairs, "real_minus_placebo"),
}, names=["quantity"]).swaplevel().sort_index()
display_table(placebo_summary, caption="Real and placebo contrasts on pairs with placebo dates")

In [ ]:
# Drawing one placebo date per pair to see how large a mean arises by chance
def placebo_randomization(contrasts, real_mean, draws=BOOTSTRAP_DRAWS, seed=RANDOM_SEED):
    groups = [g.to_numpy() for _, g in contrasts.groupby(PAIR_KEYS)["placebo_contrast"]]
    sizes = np.array([len(g) for g in groups])
    padded = np.full((len(groups), sizes.max()), np.nan)
    for i, g in enumerate(groups):
        padded[i, :len(g)] = g
    rng = np.random.default_rng(seed)
    picks = (rng.random((draws, len(groups))) * sizes).astype(int)
    null_means = padded[np.arange(len(groups)), picks].mean(axis=1)
    p_value = (1 + np.sum(null_means >= real_mean)) / (draws + 1)
    return null_means, p_value


placebo_real_mean = placebo_pairs["primary_contrast"].mean()
placebo_null_means, placebo_p_value = placebo_randomization(
    placebo_contrasts.merge(placebo_pairs[PAIR_KEYS], on=PAIR_KEYS), placebo_real_mean)

display_table(pd.DataFrame({
    "value": [placebo_real_mean, placebo_null_means.mean(), np.percentile(placebo_null_means, 2.5),
              np.percentile(placebo_null_means, 97.5), placebo_p_value]},
    index=["Real mean (covered pairs)", "Placebo mean of means", "Placebo 2.5th percentile",
           "Placebo 97.5th percentile", "One-sided randomization p-value"]), decimals=3)

fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
ax.hist(placebo_null_means, bins=40, color="#D9DDE2", edgecolor="white", label="Placebo means")
ax.axvline(placebo_real_mean, color="#10707B", linewidth=2, label="Real-event mean")
ax.axvline(0, color="#555555", linestyle="--", linewidth=1)
ax.set_xlabel("Mean contrast across covered pairs (pickups per hour)")
ax.set_ylabel("Random draws")
ax.set_title("How unusual is the real contrast compared with placebo dates?", loc="left", pad=12)
ax.legend(fontsize=9)
sns.despine(ax=ax)
plt.show()

Placebo dates are available for **338 of 368 pairs, or 91.8%**, with a median of three eligible dates. On those same pairs, the real contrast is **18.02**, the placebo mean is **1.62 [−0.53, 3.76]**, and real minus placebo is **16.41 [10.80, 22.01]** pickups/hour.

The real mean lies well beyond the sampled placebo means. The displayed value 0.000 is rounded: the saved exact value is 0.0005, the minimum possible with the code's correction and 2,000 draws. It is not a zero probability.

The draws select dates independently across pairs even when they share incidents, zones or calendar dates. I therefore treat the histogram as a descriptive separation check, not a formal randomization test. The clustered table also holds fitted historical and weather adjustments fixed. I keep 18.06 as the primary estimate rather than replacing it with real minus placebo.

I read the placebo draws as a diagnostic. Drawing dates separately for each pair can miss dependence between pairs that share an incident or a comparison zone.


## 9. Checking nearby comparison zones

A nearby comparison may also respond to the disruption. Pickups could rise there as passengers walk away from affected stations, or change because drivers relocate. Depending on the response, this could raise or lower the treated–comparison contrast.

I exclude adjacent or very nearby comparison polygons and select alternatives using the existing historical scores and rules. The code buffers both polygons by 50 feet before testing intersection, so the operational screen can include gaps of roughly 100 feet, not only shared borders.

I estimate both versions on identical supported pairs. A change would show sensitivity to the comparison restriction, but would not identify a spillover mechanism by itself.

I use TLC zone boundaries from `data/external/tlc_zones.geojson` to identify neighboring zones.


In [ ]:
def load_zone_adjacency(path):
    """Listing zone pairs whose polygons touch or lie within the buffer distance."""
    try:
        import geopandas as gpd
    except ImportError:
        print("geopandas is not installed; the spillover check is skipped.")
        return None
    if not Path(path).is_file():
        print(f"Taxi-zone shapefile not found at {path}; the spillover check is skipped.")
        return None
        # Standardizing zone identifiers before measuring distances in feet
    zones = gpd.read_file(path).rename(
        columns={"locationid": "LocationID"}
    )
    zones["LocationID"] = pd.to_numeric(
        zones["LocationID"], errors="raise"
    ).astype(int)
    zones = zones[["LocationID", "geometry"]].to_crs(2263)
    zones = zones.dissolve(by="LocationID").reset_index()
    buffered = zones.assign(geometry=zones.buffer(ADJACENCY_BUFFER_FT))
    joined = gpd.sjoin(buffered, buffered, predicate="intersects")
    adjacency = (joined[["LocationID_left", "LocationID_right"]]
                 .rename(columns={"LocationID_left": "LocationID", "LocationID_right": "comparison_zone_id"})
                 .astype(int).query("LocationID != comparison_zone_id").drop_duplicates())
    return adjacency


def estimate_primary(matches):
    """Running the primary specification end to end for a given set of matches."""
    event_hours, _, _ = build_did_sample(episode_records, matching_treated_hours, matches,
                                         preperiod_hours, research_panel)
    pair_estimates, _ = calculate_pair_contrasts(event_hours)
    coverage, _, _ = check_weekday_hour_support(research_panel, matching_candidates, matches, event_hours)
    _, adjusted_hours, adjusted_pairs = adjust_historical_gaps(
        research_panel, matching_candidates, matches, event_hours, pair_estimates, coverage)
    data, _, _, model = fit_weather_did(adjusted_hours, "adjusted_gap")
    return data, model


zone_adjacency = load_zone_adjacency(TAXI_ZONE_SHAPEFILE)
spillover_table = None

if zone_adjacency is not None:
    # How many primary pairs used at least one adjacent comparison?
    used_adjacent = preliminary_matches.merge(primary_pairs[PAIR_KEYS], on=PAIR_KEYS).merge(
        zone_adjacency.assign(adjacent=True), on=["LocationID", "comparison_zone_id"], how="left")
    pairs_with_adjacent = used_adjacent.groupby(PAIR_KEYS)["adjacent"].any().sum()

    nonadjacent_matches = select_matches(matching_scores, excluded_comparisons=zone_adjacency)

    # Estimating both versions on the pairs both can support
    nonadjacent_data, _ = estimate_primary(nonadjacent_matches)
    common_ids = set(nonadjacent_data["event_zone_id"]) & set(primary_data["event_zone_id"])
    common_pairs = primary_pairs.assign(
        event_zone_id=primary_pairs["episode_key"].astype(str) + ":" + primary_pairs["LocationID"].astype(str)
    ).loc[lambda d: d["event_zone_id"].isin(common_ids), PAIR_KEYS]

    same_data, same_model = estimate_primary(preliminary_matches.merge(common_pairs, on=PAIR_KEYS))
    adj_data, adj_model = estimate_primary(nonadjacent_matches.merge(common_pairs, on=PAIR_KEYS))
    spillover_table = pd.DataFrame([
        model_row("Primary matches (common pairs)", same_model, same_data),
        model_row("Adjacent comparisons excluded", adj_model, adj_data),
    ]).set_index("specification")
    print(f"Primary pairs using at least one adjacent comparison: {pairs_with_adjacent} of {len(primary_pairs)}")
    display_table(spillover_table, caption="Spillover check on identical pairs")

**101 of 368 primary pairs** used at least one adjacent comparison. The common supported sample contains 361 pairs from 159 incidents. Its estimate changes from **18.21 [12.14, 24.28]** with the original matches to **18.08 [11.88, 24.28]** after adjacent comparisons are excluded.

The difference is only 0.13 pickups/hour. The result is not strongly dependent on these nearby comparisons within this sample. It does not rule out spillovers farther away or changes in driver supply. I will retain the primary matches and report this as a supporting check.

## 10. Checking influential incidents and the size of the result

I want to know whether one incident or a few extreme pair contrasts explain the overall mean. I recompute the average after dropping each of the 163 incidents in turn and compare the mean with the median and a 10% trimmed mean. These calculations keep the fitted adjustments fixed; they do not rerun matching or refit the full model.

I also express the total pair contrast relative to a constructed total of expected pickups, calculated as observed during-period mean pickups minus the pair contrast. I resample incidents for its bootstrap interval. This percentage is secondary because its denominator is estimated and can be unrealistic for individual pairs.

In [ ]:
# Leave-one-incident-out means
incident_ids = primary_pairs["event_id"].unique()
leave_one_out = pd.Series(
    {e: primary_pairs.loc[primary_pairs["event_id"].ne(e), "primary_contrast"].mean() for e in incident_ids},
    name="mean_without_incident")
most_influential = (leave_one_out - primary_pairs["primary_contrast"].mean()).abs().idxmax()

# Relative effect with an incident bootstrap
relative = primary_pairs.assign(
    expected_during=primary_pairs["treated_total_pickups_during"] - primary_pairs["primary_contrast"])
incident_sums = relative.groupby("event_id")[["primary_contrast", "expected_during"]].sum()
rng = np.random.default_rng(RANDOM_SEED)
draws = rng.integers(0, len(incident_sums), size=(BOOTSTRAP_DRAWS, len(incident_sums)))
sums = incident_sums.to_numpy()
boot_ratio = 100 * sums[draws, 0].sum(axis=1) / sums[draws, 1].sum(axis=1)
relative_effect = 100 * incident_sums["primary_contrast"].sum() / incident_sums["expected_during"].sum()

fragility = pd.DataFrame({"value": {
    "Mean primary contrast": primary_pairs["primary_contrast"].mean(),
    "Median primary contrast": primary_pairs["primary_contrast"].median(),
    "10% trimmed mean": trim_mean(primary_pairs["primary_contrast"], 0.10),
    "Leave-one-incident-out minimum": leave_one_out.min(),
    "Leave-one-incident-out maximum": leave_one_out.max(),
    "Relative effect (% of expected pickups)": relative_effect,
    "Relative effect 95% bootstrap lower": np.percentile(boot_ratio, 2.5),
    "Relative effect 95% bootstrap upper": np.percentile(boot_ratio, 97.5),
    "Pairs with expected pickups below 5 per hour": int(relative["expected_during"].lt(5).sum()),
    "Pairs with negative expected pickups": int(relative["expected_during"].lt(0).sum()),
}})
display_table(fragility, decimals=2, caption="Influence, robustness and relative size")
print("Most influential incident:", most_influential)

The mean is **18.06**, compared with a median of **11.93** and a 10% trimmed mean of **13.94** pickups/hour. Larger positive contrasts pull the average upward, but the middle and trimmed summaries remain positive. Their signs alone do not supply confidence intervals.

Leaving out one incident at a time gives means from **16.94 to 18.77**. No single incident removes the positive average under the fixed fitted adjustments. Incident `201488` produces the largest absolute change in this check.

The constructed relative increase is **11.23%**, with a bootstrap interval of **8.20% to 14.40%**. Two pairs have negative expected pickup values, which are impossible as counts. I will keep pickups per hour as the main unit and describe the percentage as approximate. The bootstrap does not include refitting uncertainty.

## 11. Trying a different comparison

### Using the affected zone's own history

Matched DiD compares zones at the same actual hours. As a sensitivity analysis, I instead predict each treated zone's usual pickups from its clean preceding eight weeks. The model uses weekday–hour means and linear responses to weather, unofficial holidays and recorded sports/concert windows.

I then compare observed-minus-predicted pickups during loss with the same error before the event. Subtracting the baseline error helps account for a level mismatch, but cannot remove every event-time prediction error. This approach has no simultaneous comparison zone.

The code checks whether the historical design can estimate the required predictions. That is an algebraic support check, not proof of prediction accuracy or adequate examples of every weather intensity. Official holidays remain excluded, and event flags are proxies rather than direct crowd measurements.

In [ ]:
def prepare_same_zone_data(panel, candidates, event_hours):
    """Collecting each pair's clean history and event hours with reference features."""
    values = ["hour_local", "total_pickups", *REFERENCE_FEATURES, "is_holiday"]
    assert set(values).issubset(panel.columns), "Required features are missing."

    events = event_hours[PAIR_KEYS + ["zone_borough", "event_id", "period", "hour_local"]].merge(
        panel[["LocationID", *values]], on=["LocationID", "hour_local"], how="left", validate="many_to_one")
    assert events[["total_pickups", *REFERENCE_FEATURES]].notna().all().all()

    clean = (panel["passes_initial_quality"].eq(True) & panel["announced_minutes_all"].eq(0)
             & panel["episodes_all"].eq(0) & panel["background_route_lost_minutes"].eq(0)
             & panel["is_holiday"].eq(False)
             & panel[["total_pickups", *REFERENCE_FEATURES]].notna().all(axis=1))
    histories = dict(tuple(panel.loc[clean, ["LocationID", *values]].groupby("LocationID")))

    windows = events[PAIR_KEYS + ["zone_borough"]].drop_duplicates().merge(
        candidates[PAIR_KEYS + ["history_start_local", "history_end_local"]], on=PAIR_KEYS, validate="one_to_one")
    parts = []
    for row in windows.itertuples(index=False):
        zone = histories[row.LocationID]
        part = zone.loc[zone["hour_local"].ge(row.history_start_local)
                        & zone["hour_local"].lt(row.history_end_local)].copy()
        assert not part.empty
        part["episode_key"], part["zone_borough"] = row.episode_key, row.zone_borough
        parts.append(part)
    history = pd.concat(parts, ignore_index=True)

    for data in [history, events]:
        data["weekday_hour"] = data["hour_local"].dt.dayofweek * 24 + data["hour_local"].dt.hour
    return history, events


def fit_linear_reference(train, target):
    """Weekday-hour means plus linear feature responses, fitted on train only."""
    scales = pd.Series(FEATURE_SCALES)
    x_train = train[REFERENCE_FEATURES].astype(float).div(scales)
    x_target = target[REFERENCE_FEATURES].astype(float).div(scales)
    clock = train["weekday_hour"]

    # Removing weekday-hour means before estimating the other responses
    x_means = x_train.groupby(clock).mean()
    y_means = train.groupby(clock)["total_pickups"].mean()
    x_within = (x_train - x_train.groupby(clock).transform("mean")).to_numpy()
    y_within = (train["total_pickups"] - train.groupby(clock)["total_pickups"].transform("mean")).to_numpy()
    beta, _, rank, _ = np.linalg.lstsq(x_within, y_within, rcond=None)

    baseline = y_means.reindex(target["weekday_hour"]).to_numpy()
    centered = np.nan_to_num(x_target.to_numpy() - x_means.reindex(target["weekday_hour"]).to_numpy(), nan=0.0)

    # Flagging predictions that need responses the history cannot identify
    projection = np.linalg.pinv(x_within) @ x_within
    unsupported = centered - centered @ projection
    supported = np.isfinite(baseline) & (
        np.linalg.norm(unsupported, axis=1) <= 1e-7 * (1 + np.linalg.norm(centered, axis=1)))
    prediction = baseline + centered @ beta
    prediction[~supported] = np.nan
    return prediction, supported, int(rank)


def fit_same_zone_reference(history, events):
    """Comparing same-zone prediction errors before and during service loss."""
    history_groups = dict(tuple(history.groupby(PAIR_KEYS)))
    predictions, reviews = [], []
    for pair, event in events.groupby(PAIR_KEYS):
        train = history_groups[pair]
        event = event.copy().reset_index(drop=True)
        assert train["hour_local"].max() < event["hour_local"].min()
        expected, supported, rank = fit_linear_reference(train, event)
        event["expected_pickups"] = expected
        event["pair_supported"] = bool(supported.all())
        event["prediction_error"] = event["total_pickups"] - event["expected_pickups"]
        predictions.append(event)
        reviews.append({"episode_key": pair[0], "LocationID": pair[1],
                        "zone_borough": event["zone_borough"].iloc[0],
                        "historical_hours": len(train), "identified_dimensions": rank,
                        "pair_supported": bool(supported.all())})

    predicted = pd.concat(predictions, ignore_index=True)
    review = pd.DataFrame(reviews)
    supported = predicted.loc[predicted["pair_supported"]]
    periods = supported.groupby(PAIR_KEYS + ["period"])["prediction_error"].mean().unstack("period")
    assert periods[["before", "during"]].notna().all().all()
    periods["same_zone_contrast"] = periods["during"] - periods["before"]
    pair_results = supported[PAIR_KEYS + ["zone_borough", "event_id"]].drop_duplicates().merge(
        periods[["same_zone_contrast"]].reset_index(), on=PAIR_KEYS, validate="one_to_one")

    coverage = review.groupby("zone_borough").agg(
        candidate_pairs=("pair_supported", "size"), supported_pairs=("pair_supported", "sum"))
    coverage.loc["Total"] = coverage.sum()
    return predicted, pair_results, coverage


same_zone_history, same_zone_events = prepare_same_zone_data(
    research_panel, matching_candidates, historical_adjusted_hours)
same_zone_predictions, same_zone_pairs, same_zone_coverage = fit_same_zone_reference(
    same_zone_history, same_zone_events)
display_table(same_zone_coverage, caption="Same-zone models: pairs with supported predictions")
same_zone_summary = clustered_mean_summary(same_zone_pairs, "same_zone_contrast", "estimate")
display_table(same_zone_summary, caption="Same-zone pickup contrasts")

### Would a tree model give me a different same-zone baseline?

I tried XGBoost because the relationship between weather, time of day, and pickups may bend in ways a linear model cannot capture. For each pair, I train on the first six clean weeks, compare four settings on the last two weeks, then refit the selected setting on all eight weeks. I vary tree depth and minimum child weight while keeping the other settings fixed.

Both models are scored on the same validation hours. XGBoost gets to use those scores to pick its settings, so its reported validation error can look better than it would on fresh data. I use this check to decide how much weight to give the flexible baseline.


In [ ]:
def run_xgboost_appendix(history, events, candidates):
    """Validating linear vs tuned XGBoost references, then estimating XGBoost contrasts."""
    import xgboost as xgb
    from sklearn.model_selection import ParameterGrid

    configs = list(ParameterGrid(XGB_GRID))
    windows = candidates[PAIR_KEYS + ["history_end_local"]].copy()
    windows["validation_start"] = windows["history_end_local"] - pd.DateOffset(weeks=XGB_VALIDATION_WEEKS)
    history = history.merge(windows[PAIR_KEYS + ["validation_start"]], on=PAIR_KEYS, validate="many_to_one")

    def tree_features(data):
        result = data[REFERENCE_FEATURES].astype("float32").copy()
        result["weekday"] = data["hour_local"].dt.dayofweek.to_numpy()
        result["hour_of_day"] = data["hour_local"].dt.hour.to_numpy()
        return result

    def score(pair, borough, model, error):
        return {"episode_key": pair[0], "LocationID": pair[1], "zone_borough": borough, "model": model,
                "mae": np.abs(error).mean(), "mse": np.square(error).mean(), "bias": error.mean()}

    history_groups = dict(tuple(history.groupby(PAIR_KEYS)))
    scores, predictions = [], []
    for number, (pair, event) in enumerate(events.groupby(PAIR_KEYS), start=1):
        group = history_groups[pair]
        borough = event["zone_borough"].iloc[0]
        train = group.loc[group["hour_local"].lt(group["validation_start"])]
        valid = group.loc[group["hour_local"].ge(group["validation_start"])].reset_index(drop=True)

        linear_valid, supported, _ = fit_linear_reference(train, valid)
        valid, linear_valid = valid.loc[supported], linear_valid[supported]
        if valid.empty:
            continue
        y_valid = valid["total_pickups"].to_numpy()
        scores.append(score(pair, borough, "Linear", linear_valid - y_valid))

        # Choosing settings on the validation window only
        best = None
        for i, config in enumerate(configs):
            model = xgb.XGBRegressor(**XGB_BASE_PARAMS, **config)
            model.fit(tree_features(train), train["total_pickups"])
            error = model.predict(tree_features(valid)) - y_valid
            rmse = np.sqrt(np.mean(error ** 2))
            if best is None or rmse < best[0]:
                best = (rmse, i, error)
        scores.append(score(pair, borough, "Tuned XGBoost (optimistic)", best[2]))

        # Refitting on the full history and predicting event hours
        final = xgb.XGBRegressor(**XGB_BASE_PARAMS, **configs[best[1]])
        final.fit(tree_features(group), group["total_pickups"])
        if number == 1:
            device = json.loads(final.get_booster().save_config())["learner"]["generic_param"]["device"]
            print("XGBoost training device:", device)
        event = event.copy()
        event["xgb_error"] = event["total_pickups"] - final.predict(tree_features(event))
        predictions.append(event)
        if number % 50 == 0:
            print(f"Completed {number} pairs", flush=True)

    scores = pd.DataFrame(scores)
    validation = scores.groupby(["zone_borough", "model"]).agg(
        pairs=("mae", "size"), mae=("mae", "mean"), mse=("mse", "mean"), bias=("bias", "mean"))
    total = scores.groupby("model").agg(pairs=("mae", "size"), mae=("mae", "mean"),
                                        mse=("mse", "mean"), bias=("bias", "mean"))
    total.index = pd.MultiIndex.from_product([["Total"], total.index], names=["zone_borough", "model"])
    validation = pd.concat([validation, total])
    validation["rmse"] = np.sqrt(validation.pop("mse"))

    predicted = pd.concat(predictions, ignore_index=True)
    periods = predicted.groupby(PAIR_KEYS + ["period"])["xgb_error"].mean().unstack("period")
    contrasts = predicted[PAIR_KEYS + ["zone_borough", "event_id"]].drop_duplicates().merge(
        (periods["during"] - periods["before"]).rename("xgb_contrast").reset_index(),
        on=PAIR_KEYS, validate="one_to_one")
    return validation, contrasts


if RUN_XGBOOST_APPENDIX:
    xgb_validation, xgb_pairs = run_xgboost_appendix(same_zone_history, same_zone_events, matching_candidates)
    display_table(xgb_validation, caption="Validation errors on the last two clean weeks")
    xgb_summary = clustered_mean_summary(xgb_pairs, "xgb_contrast", "estimate")
    display_table(xgb_summary, caption="Same-zone contrasts with tuned XGBoost references")

Linear has lower overall validation **MAE (20.60 versus 21.34)** and **RMSE (35.14 versus 35.75)** than tuned XGBoost, even though the latter's validation score benefits from selection. These are pair-averaged validation summaries, not independent test errors.

XGBoost gives an overall contrast of **14.53 [7.80, 21.27]**, close to the linear same-zone **14.81 [7.94, 21.68]**. Brooklyn and Queens stay positive, while Bronx and Manhattan intervals include zero. More model flexibility does not materially change that story.

I will keep linear as the preferred historical reference. The saved output confirms `cuda:0` training. The CPU-input/GPU prediction fallback warning concerns prediction performance, not evidence of a failed fit. I will stop the model search here; agreement between these models still shares their historical-data limitations.

### Bringing the specifications together

All **368 pairs** pass the implemented same-zone support check. The same-zone contrast is **14.81 pickups/hour [7.94, 21.68]**, smaller than the primary 18.06 but still positive overall.

Brooklyn and Queens have positive intervals under both approaches. The Bronx same-zone interval includes zero, unlike its matched borough regression, and Manhattan remains inconclusive. This is why I place more confidence in the consistency of Brooklyn and Queens than in a general positive claim for every borough.

The table below puts the related specifications together. Sample sizes differ in some rows. Agreement shows sensitivity to selected choices, not independent replication, and the range of estimates is not a confidence interval.

In [ ]:
def mean_row(label, summary, estimate_column="estimate"):
    row = summary.loc["Total"]
    return {"specification": label, "episode_zone_pairs": int(row["episode_zone_pairs"]),
            "incidents": int(row["incidents"]), "contrast_pickups_per_hour": row[estimate_column],
            "ci95_lower": row["ci95_lower"], "ci95_upper": row["ci95_upper"]}


weather_only_data, _, _, weather_only_model = fit_weather_did(did_event_hours_raw, "pickup_gap")
borough_total = borough_summary.loc["Total"]

sensitivity_rows = [
    mean_row("Raw matched DiD", clustered_mean_summary(did_pair_estimates, "did_pickups_per_hour")),
    model_row("Weather only", weather_only_model, weather_only_data),
    mean_row("Weekday–hour adjustment only",
             clustered_mean_summary(historical_adjusted_pairs, "historical_adjusted_contrast")),
    model_row("PRIMARY: weekday–hour + weather", primary_model, primary_data),
    model_row("Primary, two-way clustered", primary_model_two_way, primary_data),
    {"specification": "Primary, borough-varying (pair-weighted)",
     "episode_zone_pairs": int(borough_total["episode_zone_pairs"]), "incidents": int(borough_total["incidents"]),
     "contrast_pickups_per_hour": borough_total["contrast_pickups_per_hour"],
     "ci95_lower": borough_total["ci95_lower"], "ci95_upper": borough_total["ci95_upper"]},
    mean_row("Same-zone linear reference", same_zone_summary),
]
if spillover_table is not None:
    sensitivity_rows.append({"specification": "Adjacent comparisons excluded (common pairs)",
                             **spillover_table.loc["Adjacent comparisons excluded"].to_dict()})

sensitivity_table = pd.DataFrame(sensitivity_rows).set_index("specification")
display_table(sensitivity_table, caption="Overall pickup contrast across specifications")

fig, ax = plt.subplots(figsize=(9, 0.55 * len(sensitivity_table) + 1.5), constrained_layout=True)
y = np.arange(len(sensitivity_table))
est = sensitivity_table["contrast_pickups_per_hour"].to_numpy()
err = np.vstack([est - sensitivity_table["ci95_lower"].to_numpy(),
                 sensitivity_table["ci95_upper"].to_numpy() - est])
colors = ["#10707B" if s.startswith("PRIMARY") else "#4477AA" for s in sensitivity_table.index]
for i in range(len(y)):
    ax.errorbar(est[i], y[i], xerr=err[:, [i]], fmt="o", color=colors[i], capsize=3)
ax.axvline(0, color="#555555", linestyle="--", linewidth=1)
ax.set_yticks(y, sensitivity_table.index); ax.invert_yaxis()
ax.set_xlabel("Pickup contrast per hour (95% CI)")
ax.set_title("How much does the estimate depend on the specification?", loc="left", pad=12)
ax.grid(axis="y", visible=False)
sns.despine(ax=ax)
plt.show()

Weather alone changes the raw contrast from **17.33 to 17.46**. After weekday–hour adjustment, adding weather changes **18.40 to 18.06**. These are small movements in this run; they do not establish that weather is irrelevant or perfectly controlled.

The borough-varying estimate and adjacent-comparison check stay close to the primary result. The same-zone reference gives the largest reduction among these displayed alternatives, at 14.81. Every reported overall interval in this table is above zero, but the methods share observations and can share biases.

I will keep the adjusted matched DiD as the main analysis. The other rows show where the result is stable and where the counterfactual choice matters. There is no need to average these estimates into a new effect.

## 12. Comparing with the earlier full hour run

This table preserves the earlier full-hour run as a reference. The treatment rule has intentionally changed, so downstream values are not expected to reproduce exactly. The `matches` column reports numerical agreement; `False` is not automatically a failed validation check.

The input panel remains 2,890,800 rows. Candidate pairs change from 666 to 679, matched pairs from 368 to 370, and treatment observations from 461 to 514. The adjusted analysis then excludes two unsupported pairs, leaving 368.

The row labelled `Weekday–hour adjustment only (366 pairs)` retains the old code label. Its previous value uses 366 pairs, but its current value uses 368. I leave the code and saved outputs intact and clarify the label here rather than imply an identical-sample comparison. Differences reflect the revised hours, possible changes in match eligibility and sample membership together.

In [ ]:
previous_run = {
    "Panel zone-hours": 2_890_800,
    "Candidate episode–zone pairs": 666,
    "Matched pairs": 368,
    "Matched incidents": 163,
    "Baseline observations": 1_104,
    "Treatment observations": 461,
    "Raw matched DiD": 16.92,
    "Weather only": 17.10,
    "Weekday–hour adjustment only (366 pairs)": 17.82,
    "Same-zone linear reference": 15.23,
}
current_run = {
    "Panel zone-hours": len(research_panel),
    "Candidate episode–zone pairs": len(matching_candidates),
    "Matched pairs": int(matching_summary.loc["Total", "pairs_with_matches"]),
    "Matched incidents": did_pair_estimates["event_id"].nunique(),
    "Baseline observations": int(did_sample_summary.loc[("Total", "before"), "episode_zone_hours"]),
    "Treatment observations": int(did_sample_summary.loc[("Total", "during"), "episode_zone_hours"]),
    "Raw matched DiD": sensitivity_table.loc["Raw matched DiD", "contrast_pickups_per_hour"],
    "Weather only": sensitivity_table.loc["Weather only", "contrast_pickups_per_hour"],
    "Weekday–hour adjustment only (366 pairs)":
        sensitivity_table.loc["Weekday–hour adjustment only", "contrast_pickups_per_hour"],
    "Same-zone linear reference":
        sensitivity_table.loc["Same-zone linear reference", "contrast_pickups_per_hour"],
}
checkpoints = pd.DataFrame({"previous": previous_run, "current": current_run})
checkpoints["matches"] = np.isclose(checkpoints["previous"].astype(float),
                                    checkpoints["current"].astype(float), atol=0.005)
display(checkpoints)

## 13. What I can take from this analysis

For the retained sustained-loss episodes, the results are consistent with increased completed FHV pickups. The main adjusted contrast is **18.06 pickups per hour [12.06, 24.05]**, across **368 episode–zone pairs and 163 incidents**. This is a pair-weighted estimate; incidents affecting more zones contribute more pairs.

Including qualifying 45-minute boundary hours retains more exposure without materially changing the primary estimate from the earlier 18.08. The event study shows no clear pre-anchor divergence in the reported individual intervals and a positive response afterward. Clean-date placebo contrasts are much smaller, and excluding nearby comparisons barely changes the estimate.

Brooklyn and Queens give the most consistent positive evidence across matched and same-zone approaches. The Bronx depends more on the comparison method. Manhattan remains inconclusive under the borough regression and same-zone analysis. I do not formally establish that borough effects differ from one another.

The estimate describes completed pickups during selected, matched, nonholiday weekday daytime service-loss hours. It does not measure requests that never became rides or which passengers switched modes. Spillovers, unobserved local conditions and exposure errors can remain. The reported intervals do not include all uncertainty from historical estimation, matching or prediction fitting.

I will retain matched DiD as the main result and same-zone models as supporting sensitivity analyses. Recovery is a later research question: it will require observed service evidence and a separate definition, rather than treating the end of a loss record as restored service.

To study recovery, I'd need records showing when trains actually started serving each station again and enough clean comparison hours afterward. The end of an alert gives me a time to investigate. It does not tell me when service came back.


In [ ]:
# Key results regenerated from this run
key = sensitivity_table.loc["PRIMARY: weekday–hour + weather"]
two_way = sensitivity_table.loc["Primary, two-way clustered"]
pre_hours = event_study_balanced.loc[[k for k in EVENT_STUDY_HOURS if k < -1]]
key_results = pd.DataFrame({"value": {
    "Primary contrast (pickups/hour)": key["contrast_pickups_per_hour"],
    "Incident-clustered 95% CI": f"[{key['ci95_lower']:.2f}, {key['ci95_upper']:.2f}]",
    "Two-way clustered 95% CI": f"[{two_way['ci95_lower']:.2f}, {two_way['ci95_upper']:.2f}]",
    "Relative effect (%)": relative_effect,
    "Pairs / incidents": f"{int(key['episode_zone_pairs'])} / {int(key['incidents'])}",
    "Largest |pre-anchor event-study estimate|": pre_hours["estimate"].abs().max(),
    "Pairs with placebo dates": f"{len(placebo_pairs)} of {len(primary_pairs)}",
    "Placebo randomization p-value": placebo_p_value,
    "Spillover check": ("not run" if spillover_table is None else
                        f"{spillover_table['contrast_pickups_per_hour'].iloc[0]:.2f} → "
                        f"{spillover_table['contrast_pickups_per_hour'].iloc[1]:.2f}"),
}})
display(key_results)